# Line Flat-Top Holography AutoDL Notebook v58

This version fixes structural and consistency issues identified in v55.

Key changes from v55:
1. Removed duplicate `for rel_path, text in project_files.items()` loop;
2. Reference generation uses `cfg.efficiency_floor` instead of hardcoded `0.05`;
3. `_rank_key` includes `below_floor_penalty` as primary sort key;
4. `_route_score` weights rescaled: uniformity 0.6->80.0, overlap 0.1->1000.0;
5. `efficiency_floor` unified to 0.15 across config default and route overrides;
6. Removed dead first `line_flat_top_target` definition;
7. `hybrid_init_score` now accepts and passes `efficiency_floor`;
8. `_phase_only_correct` gate removed: all routes now get phase post-correction;
9. Training `efficiency_floor` raised from 0.05 to 0.30, `flat_efficiency_weight` from 0.30 to 0.60;
10. `output_size` raised from 2048 to 4096 (4x padding for better Fourier sampling);
11. Runtime assertions added to verify ROI parameters match expectations;
12. `warm_start_external_mix_floor` set to 0.0 for flat_top routes;



In [ ]:
import importlib.util
import os
import subprocess
import sys
from packaging.version import Version


required = {
    "torch_min": "2.7.0",
    "torch_max": "2.9.0",
}


def ensure_installed(packages: list[str]) -> None:
    print("$", sys.executable, "-m", "pip", "install", *packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])


basic_missing = []
for module_name, package_name in [
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
]:
    if importlib.util.find_spec(module_name) is None:
        basic_missing.append(package_name)

if basic_missing:
    ensure_installed(basic_missing)

torch_missing = [name for name in ["torch"] if importlib.util.find_spec(name) is None]
if torch_missing:
    raise RuntimeError(
        "Missing torch. Please preinstall a GPU build of torch with CUDA 12.8 in AutoDL."
    )

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

import torch

print("torch =", torch.__version__)
print("cuda available =", torch.cuda.is_available())
print("torch cuda =", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu =", torch.cuda.get_device_name(0))

torch_version = Version(torch.__version__.split("+")[0])
if not (Version(required["torch_min"]) <= torch_version < Version(required["torch_max"])):
    raise RuntimeError(
        f"Expected torch in [{required['torch_min']}, {required['torch_max']}), got {torch.__version__}"
    )
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Please switch to a GPU environment with CUDA 12.8.")
if str(torch.version.cuda) != "12.8":
    raise RuntimeError(f"Expected CUDA 12.8, got {torch.version.cuda}")

print("Dependency check completed.")


In [ ]:
from pathlib import Path
import importlib
import os
import sys


def pick_project_root() -> Path:
    candidates = [
        Path("/root/autodl-tmp"),
        Path("/tmp"),
        Path.cwd(),
    ]
    for base in candidates:
        try:
            base.mkdir(parents=True, exist_ok=True)
            root = base / "flat_top_light_runtime_v58"
            root.mkdir(parents=True, exist_ok=True)
            probe = root / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return root
        except Exception:
            continue
    raise RuntimeError(
        "No writable runtime directory found. Please set PROJECT_ROOT manually, e.g. /root/autodl-tmp/flat_top_light_runtime_v58."
    )


PROJECT_ROOT = pick_project_root()

for name in list(sys.modules):
    if name == "ai_holography" or name.startswith("ai_holography.") or name == "lin2025_holography" or name.startswith("lin2025_holography."):
        del sys.modules[name]

project_files = {
  "ai_holography/__init__.py": "from .config import HolographyConfig\nfrom .pipeline import AIHolographyPipeline\nfrom .runner import HybridHolographyRunner\n",
  "ai_holography/calibration.py": "from __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\ndef load_phase_correction(path: str | Path | None, size: int, device: str) -> torch.Tensor:\n    if path is None:\n        return torch.zeros((size, size), dtype=torch.float32, device=device)\n    path = Path(path)\n    if not path.exists():\n        return torch.zeros((size, size), dtype=torch.float32, device=device)\n    if path.suffix == \".npy\":\n        arr = np.load(path)\n    else:\n        arr = np.loadtxt(path)\n    arr = np.asarray(arr, dtype=np.float32)\n    if arr.shape != (size, size):\n        raise ValueError(f\"Correction phase shape {arr.shape} does not match expected {(size, size)}\")\n    return torch.tensor(arr, dtype=torch.float32, device=device)\n\n",
  "ai_holography/camera_loop.py": "from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\n@dataclass\nclass CameraLoopConfig:\n    enabled: bool = False\n    feedback_gain: float = 0.2\n    max_rounds: int = 3\n    measured_intensity_path: str | None = None\n\n\ndef load_measured_intensity(path: str | None, expected_shape: tuple[int, int]) -> torch.Tensor | None:\n    if path is None:\n        return None\n    p = Path(path)\n    if not p.exists():\n        return None\n    if p.suffix == \".npy\":\n        arr = np.load(p)\n    else:\n        arr = np.loadtxt(p)\n    arr = np.asarray(arr, dtype=np.float32)\n    if arr.shape != expected_shape:\n        raise ValueError(f\"Measured intensity shape {arr.shape} does not match expected {expected_shape}\")\n    return torch.tensor(arr, dtype=torch.float32)\n\n\ndef save_measured_intensity(path: str | Path, intensity: torch.Tensor) -> None:\n    p = Path(path)\n    p.parent.mkdir(parents=True, exist_ok=True)\n    np.save(p, intensity.detach().cpu().numpy())\n\n\ndef camera_feedback_weight(measured_intensity: torch.Tensor, target_intensity: torch.Tensor) -> torch.Tensor:\n    measured = measured_intensity.to(target_intensity.device)\n    measured = measured / measured.mean().clamp_min(1e-9)\n    target = target_intensity / target_intensity.mean().clamp_min(1e-9)\n    ratio = target / measured.clamp_min(1e-6)\n    return ratio\n\n\ndef update_target_from_measurement(\n    previous_target_amp: torch.Tensor,\n    measured_intensity: torch.Tensor,\n    gain: float,\n) -> torch.Tensor:\n    prev_i = previous_target_amp.square()\n    measured = measured_intensity.to(previous_target_amp.device)\n    measured = measured / measured.max().clamp_min(1e-9)\n    prev_norm = prev_i / prev_i.max().clamp_min(1e-9)\n    correction = prev_norm - measured\n    new_goal = torch.clamp(prev_norm + gain * correction, min=0.0)\n    return torch.sqrt(new_goal)\n",
  "ai_holography/config.py": "from dataclasses import dataclass, field\nfrom datetime import datetime\nfrom pathlib import Path\n\n\n@dataclass\nclass HolographyConfig:\n    preset: str = \"bowman_lg\"\n    slm_size: int = 1024\n    output_size: int = 4096\n    wavelength_m: float = 795e-9\n    focal_length_m: float = 0.2\n    slm_pixel_pitch_m: float = 12.5e-6\n    magnification: float = 1.0\n    target_shift_x_m: float = 0.0\n    target_shift_y_m: float = 0.0\n    beam_sigma_x: float = 160.0\n    beam_sigma_y: float = 160.0\n    target_sigma: float = 28.0\n    vortex_charge: int = 1\n    target_type: str = \"target_lg\"\n    phase_type: str = \"phase_spinning_continuous\"\n    weight_type: str = \"gaussian_top_round\"\n    training_target_types: tuple[str, ...] | None = None\n    target_center_x: float | None = None\n    target_center_y: float | None = None\n    slm_phase_correction_path: Path | None = None\n    reference_phase_path: Path | None = None\n    reference_phase_mix: float = 0.75\n    init_candidate_sources: tuple[str, ...] = (\"neural\", \"reference\", \"lin2025\", \"warm_start\")\n    lin2025_checkpoint_dir: Path = Path(\"lin2025_holography\") / \"checkpoints\"\n    lin2025_checkpoint: Path | None = None\n    lin2025_mix: float = 0.6\n    lin2025_primary: bool = False\n    lin2025_reference_weak_mix: float = 0.35\n    warm_start_external_mix_floor: float = 0.25\n    init_score_core_uniformity_weight: float = 1.0\n    init_score_core_phase_weight: float = 1.0\n    init_score_efficiency_weight: float = 0.5\n    init_score_uniformity_weight: float = 0.2\n    reference_output_size: int = 6876\n    roi_diameter: float = 128.0\n    roi_softness: float = 8.0\n    flat_top_order: float = 8.0\n    flat_top_edge_softness: float = 0.12\n    line_flat_top_sigma_x: float = 3.0\n    line_flat_top_sigma_y: float = 3.0\n    weight_threshold: float = 1e-4\n    init_tilt: float = -1.5707963267948966\n    init_aspect: float = 0.5\n    init_curvature: float = 0.003\n    init_angle: float = 0.7853981633974483\n    init_cone: float = 0.0\n    phase_residual_scale: float = 1.5707963267948966\n    learning_rate: float = 3e-2\n    iterations: int = 600\n    multiscale_levels: tuple[int, ...] = (256, 512, 1024)\n    stage_iterations: tuple[int, ...] = (120, 90, 60)\n    early_stop_patience: int = 20\n    early_stop_min_delta: float = 1e-5\n    target_overlap: float = 0.9993\n    refine_with_lbfgs: bool = True\n    lbfgs_steps: int = 40\n    skip_lbfgs_if_target_met: bool = True\n    hybrid_cg_maxiter: int = 60\n    polish_stage1_maxiter: int = 20\n    polish_stage2_maxiter: int = 15\n    polish_stage3_maxiter: int = 8\n    polish_enable_phase_priority: bool = True\n    polish_phase_priority_core_uniformity_weight: float = 0.80\n    polish_phase_priority_core_phase_weight: float = 1.20\n    polish_phase_priority_uniformity_weight: float = 0.10\n    polish_phase_priority_efficiency_weight: float = 0.12\n    polish_phase_priority_phase_weight: float = 0.30\n    polish_overlap_weight: float = 1.0\n    polish_intensity_weight: float = 0.2\n    polish_phase_weight: float = 0.15\n    polish_uniformity_weight: float = 0.25\n    polish_efficiency_weight: float = 0.15\n    polish_smoothness_weight: float = 1e-4\n    efficiency_floor: float = 0.15\n    phase_post_correction_enabled: bool = True\n    phase_post_correction_steps: int = 12\n    phase_post_correction_lr: float = 8e-3\n    phase_post_correction_max_delta: float = 0.2\n    phase_post_correction_core_uniformity_weight: float = 0.15\n    phase_post_correction_core_phase_weight: float = 0.50\n    phase_post_correction_efficiency_weight: float = 0.20\n    phase_post_correction_uniformity_weight: float = 0.05\n    warm_start_similarity_threshold: float = 0.995\n    warm_start_min_mix: float = 0.15\n    warm_start_max_mix: float = 0.9\n    warm_start_candidate_threshold: float = 0.7\n    warm_start_allow_direct: bool = True\n    fast_init_overlap_threshold: float = 0.998\n    adaptive_weighting: bool = True\n    adaptive_weighting_alpha: float = 0.25\n    adaptive_weight_clip_min: float = 0.5\n    adaptive_weight_clip_max: float = 2.0\n    overlap_weight: float = 1.0\n    intensity_weight: float = 0.25\n    phase_weight: float = 0.15\n    uniformity_weight: float = 0.2\n    efficiency_weight: float = 0.1\n    core_uniformity_weight: float = 0.30\n    core_phase_weight: float = 0.50\n    core_region_threshold: float = 0.7\n    smoothness_weight: float = 1e-4\n    schedule_ramp_start: float = 0.35\n    schedule_ramp_end: float = 0.85\n    run_name: str = field(default_factory=lambda: datetime.now().strftime(\"%Y%m%d_%H%M%S\"))\n    run_dir: Path = field(default_factory=lambda: Path(\"ai_holography\") / \"runs\")\n    output_dir: Path | None = None\n    comparison_dir: Path | None = None\n    benchmark_dir: Path | None = None\n    checkpoint: Path | None = None\n    auto_load_best_checkpoint: bool = True\n    checkpoint_dir: Path = Path(\"ai_holography\") / \"checkpoints\"\n    reference_dir: Path = Path(\"ftl_gen\")\n    enable_reference_pretrain: bool = True\n    reference_pretrain_epochs: int = 2\n    reference_phase_loss_weight: float = 1.0\n    reference_tv_loss_weight: float = 1e-4\n    reference_physics_loss_weight: float = 0.2\n    reference_pretrain_lowres_size: int = 128\n    reference_augmented_samples: int = 64\n    reference_shift_px: float = 8.0\n    reference_scale_jitter: float = 0.12\n    reference_beam_sigma_jitter: float = 0.08\n    training_uniformity_metric_weight: float = 0.3\n    training_core_uniformity_metric_weight: float = 1.0\n    training_core_phase_metric_weight: float = 1.0\n    training_efficiency_metric_weight: float = 0.6\n    training_round_uniformity_metric_weight: float = 0.2\n    training_round_core_uniformity_metric_weight: float = 0.6\n    training_round_core_phase_metric_weight: float = 0.6\n    training_round_efficiency_metric_weight: float = 1.0\n    training_flat_uniformity_metric_weight: float = 0.1\n    training_flat_core_uniformity_metric_weight: float = 1.6\n    training_flat_core_phase_metric_weight: float = 1.6\n    training_flat_efficiency_metric_weight: float = 0.5\n    training_metric_patience: int = 2\n    training_metric_min_delta: float = 1e-4\n    device: str = \"cpu\"\n    feedback_target_gain: float = 0.15\n\n    def __post_init__(self) -> None:\n        self.run_dir = Path(self.run_dir) / self.run_name\n        if self.output_dir is None:\n            self.output_dir = self.run_dir / \"outputs\"\n        else:\n            self.output_dir = Path(self.output_dir)\n        if self.comparison_dir is None:\n            self.comparison_dir = self.run_dir / \"comparison\"\n        else:\n            self.comparison_dir = Path(self.comparison_dir)\n        if self.benchmark_dir is None:\n            self.benchmark_dir = self.run_dir / \"benchmarks\"\n        else:\n            self.benchmark_dir = Path(self.benchmark_dir)\n",
  "ai_holography/experimental_heuristics.py": "from __future__ import annotations\n\nimport math\n\nimport torch\n\n\ndef focal_plane_pixel_pitch(\n    wavelength_m: float,\n    focal_length_m: float,\n    output_pixels: int,\n    slm_pixel_pitch_m: float,\n    magnification: float = 1.0,\n) -> float:\n    return wavelength_m * focal_length_m / (output_pixels * slm_pixel_pitch_m * magnification)\n\n\ndef target_shift_pixels(\n    dx_m: float,\n    dy_m: float,\n    wavelength_m: float,\n    focal_length_m: float,\n    output_shape: tuple[int, int],\n    slm_pixel_pitch_m: float,\n    magnification: float = 1.0,\n) -> tuple[float, float]:\n    pitch_y = focal_plane_pixel_pitch(\n        wavelength_m=wavelength_m,\n        focal_length_m=focal_length_m,\n        output_pixels=output_shape[0],\n        slm_pixel_pitch_m=slm_pixel_pitch_m,\n        magnification=magnification,\n    )\n    pitch_x = focal_plane_pixel_pitch(\n        wavelength_m=wavelength_m,\n        focal_length_m=focal_length_m,\n        output_pixels=output_shape[1],\n        slm_pixel_pitch_m=slm_pixel_pitch_m,\n        magnification=magnification,\n    )\n    return dx_m / pitch_x, dy_m / pitch_y\n\n\ndef physical_phase_guess(\n    slm_shape: tuple[int, int],\n    dx_m: float,\n    dy_m: float,\n    wavelength_m: float,\n    focal_length_m: float,\n    slm_pixel_pitch_m: float,\n    magnification: float,\n    aspect: float,\n    curvature: float,\n    cone: float,\n    device: str,\n) -> torch.Tensor:\n    rows, cols = slm_shape\n    x = torch.arange(cols, dtype=torch.float32, device=device) - cols / 2.0\n    y = torch.arange(rows, dtype=torch.float32, device=device) - rows / 2.0\n    X, Y = torch.meshgrid(x, y, indexing=\"xy\")\n\n    dx_px = dx_m / slm_pixel_pitch_m\n    dy_px = dy_m / slm_pixel_pitch_m\n    mu = math.atan2(dy_px, dx_px if abs(dx_px) > 1e-12 else 1e-12)\n    distance_px = math.sqrt(dx_px * dx_px + dy_px * dy_px)\n    tilt_scale = 2.0 * math.pi * slm_pixel_pitch_m / max(wavelength_m * focal_length_m, 1e-12)\n    D = 4.0 * tilt_scale * distance_px\n    ang = torch.tensor(mu, dtype=torch.float32, device=device)\n\n    Xn = X / max(rows, cols)\n    Yn = Y / max(rows, cols)\n    linear = D * (Xn * torch.cos(ang) + Yn * torch.sin(ang))\n    quad = 3.0 * curvature * (aspect * Xn.square() + (1.0 - aspect) * Yn.square())\n    radial = cone * torch.sqrt(Xn.square() + Yn.square())\n    return linear + quad + radial\n\n",
  "ai_holography/hybrid.py": "from __future__ import annotations\n\nimport time\n\nimport numpy as np\nimport scipy.optimize\nimport torch\n\nfrom .config import HolographyConfig\nfrom .losses import composite_loss, efficiency_weighted_overlap_loss, normalized_overlap\nfrom .propagation import FourierSLM, wrap_phase\n\n\ndef bowman_cg_refine(\n    cfg: HolographyConfig,\n    problem: dict[str, torch.Tensor],\n    init_phase: torch.Tensor,\n    maxiter: int = 120,\n    target_overlap: float | None = None,\n) -> dict[str, object]:\n    device = torch.device(cfg.device)\n    propagator = FourierSLM(cfg.slm_size, cfg.output_size).to(device)\n    beam = problem[\"beam\"]\n    target_amp = problem[\"target_amp\"]\n    target_phase = problem[\"target_phase\"]\n    weight = problem[\"weight\"]\n\n    def run_cg_stage(\n        x0: np.ndarray,\n        maxiter_stage: int,\n        objective_mode: str,\n        target_overlap_stage: float | None,\n    ) -> tuple[np.ndarray, float, int, float]:\n        best: dict[str, object] = {\n            \"phase\": x0.copy(),\n            \"loss\": float(\"inf\"),\n            \"overlap\": 0.0,\n        }\n\n        def objective_and_grad(flat_phase: np.ndarray) -> tuple[float, np.ndarray]:\n            phase = torch.tensor(\n                flat_phase.reshape(cfg.slm_size, cfg.slm_size),\n                dtype=torch.float32,\n                device=device,\n                requires_grad=True,\n            )\n            out_amp, out_phase = propagator(beam, phase)\n            overlap = normalized_overlap(target_amp, target_phase, out_amp, out_phase, weight)\n            if objective_mode == \"overlap\":\n                loss = 1e9 * (1.0 - overlap).square()\n            elif objective_mode == \"efficiency_overlap\":\n                loss = 1e9 * efficiency_weighted_overlap_loss(\n                    target_amp=target_amp,\n                    target_phase=target_phase,\n                    out_amp=out_amp,\n                    out_phase=out_phase,\n                    weight=weight,\n                    steepness=2.0,\n                )\n            elif objective_mode == \"phase_priority\":\n                loss, _ = composite_loss(\n                    target_amp=target_amp,\n                    target_phase=target_phase,\n                    out_amp=out_amp,\n                    out_phase=out_phase,\n                    weight=weight,\n                    slm_phase=phase,\n                    efficiency_floor=cfg.efficiency_floor,\n                    overlap_weight=cfg.polish_overlap_weight,\n                    intensity_weight=0.0,\n                    phase_weight=cfg.polish_phase_priority_phase_weight,\n                    uniformity_weight=cfg.polish_phase_priority_uniformity_weight,\n                    efficiency_weight=cfg.polish_phase_priority_efficiency_weight,\n                    core_uniformity_weight=cfg.polish_phase_priority_core_uniformity_weight,\n                    core_phase_weight=cfg.polish_phase_priority_core_phase_weight,\n                    core_region_threshold=cfg.core_region_threshold,\n                    smoothness_weight=cfg.polish_smoothness_weight,\n                )\n            else:\n                loss, _ = composite_loss(\n                    target_amp=target_amp,\n                    target_phase=target_phase,\n                    out_amp=out_amp,\n                    out_phase=out_phase,\n                    weight=weight,\n                    slm_phase=phase,\n                    efficiency_floor=cfg.efficiency_floor,\n                    overlap_weight=cfg.polish_overlap_weight,\n                    intensity_weight=cfg.polish_intensity_weight,\n                    phase_weight=cfg.polish_phase_weight,\n                    uniformity_weight=cfg.polish_uniformity_weight,\n                    efficiency_weight=cfg.polish_efficiency_weight,\n                    core_uniformity_weight=cfg.core_uniformity_weight,\n                    core_phase_weight=cfg.core_phase_weight,\n                    core_region_threshold=cfg.core_region_threshold,\n                    smoothness_weight=cfg.polish_smoothness_weight,\n                )\n            loss.backward()\n            grad = phase.grad.detach().cpu().numpy().astype(np.float64).ravel()\n            loss_value = float(loss.detach().cpu())\n            overlap_value = float(overlap.detach().cpu())\n            if loss_value < best[\"loss\"]:\n                best[\"loss\"] = loss_value\n                best[\"overlap\"] = overlap_value\n                best[\"phase\"] = flat_phase.copy()\n            return loss_value, grad\n\n        def func(flat_phase: np.ndarray) -> float:\n            value, _ = objective_and_grad(flat_phase)\n            return value\n\n        def grad(flat_phase: np.ndarray) -> np.ndarray:\n            _, gradient = objective_and_grad(flat_phase)\n            return gradient\n\n        callback_state = {\"steps\": 0}\n\n        def callback(flat_phase: np.ndarray) -> None:\n            callback_state[\"steps\"] += 1\n            if target_overlap_stage is not None and best[\"overlap\"] >= target_overlap_stage:\n                raise StopIteration\n\n        stage_start = time.perf_counter()\n        try:\n            res = scipy.optimize.fmin_cg(\n                f=func,\n                x0=x0,\n                fprime=grad,\n                maxiter=maxiter_stage,\n                disp=False,\n                full_output=True,\n                retall=False,\n                callback=callback,\n            )\n            result_phase = res[0]\n            result_cost = float(res[1])\n        except StopIteration:\n            result_phase = best[\"phase\"]\n            result_cost = float(best[\"loss\"])\n        stage_runtime = time.perf_counter() - stage_start\n        return np.asarray(result_phase), result_cost, callback_state[\"steps\"], stage_runtime\n\n    x0 = init_phase.detach().cpu().numpy().astype(np.float64).ravel()\n    start = time.perf_counter()\n    stage1_phase, stage1_cost, stage1_steps, stage1_runtime = run_cg_stage(\n        x0=x0,\n        maxiter_stage=min(maxiter, cfg.polish_stage1_maxiter),\n        objective_mode=\"efficiency_overlap\",\n        target_overlap_stage=target_overlap,\n    )\n    remaining = max(\n        0,\n        min(maxiter, cfg.polish_stage1_maxiter + cfg.polish_stage2_maxiter + cfg.polish_stage3_maxiter) - stage1_steps,\n    )\n    if remaining > 0:\n        stage2_phase, stage2_cost, stage2_steps, stage2_runtime = run_cg_stage(\n            x0=stage1_phase,\n            maxiter_stage=min(remaining, cfg.polish_stage2_maxiter),\n            objective_mode=\"composite\",\n            target_overlap_stage=target_overlap,\n        )\n        result_phase = stage2_phase\n        result_cost = stage2_cost\n        total_steps = stage1_steps + stage2_steps\n        stage2_used = True\n        stage2_time = stage2_runtime\n        remaining_after_stage2 = max(0, remaining - stage2_steps)\n    else:\n        result_phase = stage1_phase\n        result_cost = stage1_cost\n        total_steps = stage1_steps\n        stage2_used = False\n        stage2_time = 0.0\n        remaining_after_stage2 = 0\n    stage3_used = False\n    stage3_time = 0.0\n    if cfg.polish_enable_phase_priority and remaining_after_stage2 > 0:\n        stage3_phase, stage3_cost, stage3_steps, stage3_runtime = run_cg_stage(\n            x0=result_phase,\n            maxiter_stage=min(remaining_after_stage2, cfg.polish_stage3_maxiter),\n            objective_mode=\"phase_priority\",\n            target_overlap_stage=target_overlap,\n        )\n        result_phase = stage3_phase\n        result_cost = stage3_cost\n        total_steps += stage3_steps\n        stage3_used = True\n        stage3_time = stage3_runtime\n    runtime = time.perf_counter() - start\n\n    final_phase = wrap_phase(\n        torch.tensor(np.asarray(result_phase).reshape(cfg.slm_size, cfg.slm_size), dtype=torch.float32, device=device)\n    )\n    out_amp, out_phase = propagator(beam, final_phase)\n    overlap = normalized_overlap(target_amp, target_phase, out_amp, out_phase, weight)\n    loss, aux = composite_loss(\n        target_amp=target_amp,\n        target_phase=target_phase,\n        out_amp=out_amp,\n        out_phase=out_phase,\n        weight=weight,\n        slm_phase=final_phase,\n        efficiency_floor=cfg.efficiency_floor,\n        overlap_weight=cfg.polish_overlap_weight,\n        intensity_weight=cfg.polish_intensity_weight,\n        phase_weight=cfg.polish_phase_weight,\n        uniformity_weight=cfg.polish_uniformity_weight,\n        efficiency_weight=cfg.polish_efficiency_weight,\n        core_uniformity_weight=cfg.core_uniformity_weight,\n        core_phase_weight=cfg.core_phase_weight,\n        core_region_threshold=cfg.core_region_threshold,\n        smoothness_weight=cfg.polish_smoothness_weight,\n    )\n    metrics = {\n        \"runtime_sec\": runtime,\n        \"bowman_cost\": result_cost,\n        \"overlap\": float(overlap.detach().cpu()),\n        \"composite_loss\": float(loss.detach().cpu()),\n        \"cg_steps\": total_steps,\n        \"stage1_steps\": stage1_steps,\n        \"stage2_used\": stage2_used,\n        \"stage2_runtime_sec\": stage2_time,\n        \"stage3_used\": stage3_used,\n        \"stage3_runtime_sec\": stage3_time,\n        **aux,\n    }\n    return {\n        \"phase\": final_phase.detach().cpu(),\n        \"out_amp\": out_amp.detach().cpu(),\n        \"out_phase\": out_phase.detach().cpu(),\n        \"metrics\": metrics,\n    }\n",
  "ai_holography/losses.py": "import math\nimport torch\n\n\ndef normalized_overlap(\n    target_amp: torch.Tensor,\n    target_phase: torch.Tensor,\n    out_amp: torch.Tensor,\n    out_phase: torch.Tensor,\n    weight: torch.Tensor,\n) -> torch.Tensor:\n    numerator = torch.sum(target_amp * out_amp * weight * torch.cos(out_phase - target_phase))\n    denominator = torch.sqrt(torch.sum(target_amp.square()) * torch.sum((out_amp * weight).square())).clamp_min(1e-9)\n    return numerator / denominator\n\n\ndef weighted_intensity_loss(target_amp: torch.Tensor, out_amp: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:\n    target_intensity = target_amp.square()\n    out_intensity = out_amp.square()\n    diff = (target_intensity - out_intensity) * weight\n    return diff.square().mean()\n\n\ndef target_core_weight(target_amp: torch.Tensor, weight: torch.Tensor, threshold: float) -> torch.Tensor:\n    target_intensity = target_amp.square()\n    if torch.max(target_intensity) <= 0:\n        return torch.zeros_like(weight)\n    core = (target_intensity / torch.max(target_intensity).clamp_min(1e-9)) >= threshold\n    return weight * core.to(weight.dtype)\n\n\ndef wrapped_phase_loss(target_phase: torch.Tensor, out_phase: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:\n    phase_error = torch.atan2(torch.sin(out_phase - target_phase), torch.cos(out_phase - target_phase))\n    return (phase_error.square() * weight).mean()\n\n\ndef phase_flatness_metric(target_phase: torch.Tensor, out_phase: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:\n    phase_error = torch.atan2(torch.sin(out_phase - target_phase), torch.cos(out_phase - target_phase))\n    weighted = phase_error[weight > 0]\n    if weighted.numel() <= 1:\n        return torch.tensor(0.0, device=out_phase.device, dtype=out_phase.dtype)\n    return weighted.std(unbiased=False)\n\n\ndef uniformity_loss(target_amp: torch.Tensor, out_amp: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:\n    target_intensity = target_amp.square()\n    out_intensity = out_amp.square()\n    mask = weight > 0\n    if mask.sum() == 0:\n        return torch.tensor(0.0, device=out_amp.device)\n    target_vals = target_intensity[mask]\n    out_vals = out_intensity[mask]\n    # Compare normalized local intensity to improve uniformity inside the active region.\n    target_norm = target_vals / target_vals.mean().clamp_min(1e-9)\n    out_norm = out_vals / out_vals.mean().clamp_min(1e-9)\n    return (out_norm - target_norm).square().mean()\n\n\ndef core_uniformity_loss(\n    target_amp: torch.Tensor,\n    out_amp: torch.Tensor,\n    weight: torch.Tensor,\n    threshold: float,\n) -> torch.Tensor:\n    return uniformity_loss(target_amp, out_amp, target_core_weight(target_amp, weight, threshold))\n\n\ndef core_phase_flatness_metric(\n    target_phase: torch.Tensor,\n    out_phase: torch.Tensor,\n    target_amp: torch.Tensor,\n    weight: torch.Tensor,\n    threshold: float,\n) -> torch.Tensor:\n    return phase_flatness_metric(target_phase, out_phase, target_core_weight(target_amp, weight, threshold))\n\n\ndef core_phase_loss(\n    target_phase: torch.Tensor,\n    out_phase: torch.Tensor,\n    target_amp: torch.Tensor,\n    weight: torch.Tensor,\n    threshold: float,\n) -> torch.Tensor:\n    return wrapped_phase_loss(target_phase, out_phase, target_core_weight(target_amp, weight, threshold))\n\n\ndef efficiency_metric(weight: torch.Tensor, out_amp: torch.Tensor) -> torch.Tensor:\n    out_intensity = out_amp.square()\n    useful = torch.sum(out_intensity * (weight > 0).to(out_intensity.dtype))\n    total = torch.sum(out_intensity).clamp_min(1e-9)\n    return useful / total\n\n\ndef efficiency_loss(weight: torch.Tensor, out_amp: torch.Tensor) -> torch.Tensor:\n    return 1.0 - efficiency_metric(weight, out_amp)\n\n\ndef soft_efficiency_constraint(efficiency: torch.Tensor, floor: float) -> torch.Tensor:\n    margin = torch.relu(torch.tensor(floor, device=efficiency.device, dtype=efficiency.dtype) - efficiency)\n    return margin.square()\n\n\ndef efficiency_weighted_overlap_loss(\n    target_amp: torch.Tensor,\n    target_phase: torch.Tensor,\n    out_amp: torch.Tensor,\n    out_phase: torch.Tensor,\n    weight: torch.Tensor,\n    steepness: float = 2.0,\n) -> torch.Tensor:\n    overlap = normalized_overlap(target_amp, target_phase, out_amp, out_phase, weight)\n    eff = efficiency_metric(weight, out_amp)\n    return (1.0 - overlap).square() * torch.pow(1.0 / eff.clamp_min(1e-6), steepness)\n\n\ndef total_variation_loss(phase: torch.Tensor) -> torch.Tensor:\n    dy = phase[1:, :] - phase[:-1, :]\n    dx = phase[:, 1:] - phase[:, :-1]\n    return dx.abs().mean() + dy.abs().mean()\n\n\ndef composite_loss(\n    target_amp: torch.Tensor,\n    target_phase: torch.Tensor,\n    out_amp: torch.Tensor,\n    out_phase: torch.Tensor,\n    weight: torch.Tensor,\n    slm_phase: torch.Tensor,\n    overlap_weight: float,\n    intensity_weight: float,\n    phase_weight: float,\n    smoothness_weight: float,\n    uniformity_weight: float = 0.0,\n    efficiency_weight: float = 0.0,\n    core_uniformity_weight: float = 0.30,\n    core_phase_weight: float = 0.50,\n    core_region_threshold: float = 0.7,\n    efficiency_floor: float | None = None,\n) -> tuple[torch.Tensor, dict[str, float]]:\n    overlap = normalized_overlap(target_amp, target_phase, out_amp, out_phase, weight)\n    overlap_term = (1.0 - overlap).square()\n    intensity_term = weighted_intensity_loss(target_amp, out_amp, weight)\n    phase_term = wrapped_phase_loss(target_phase, out_phase, weight)\n    flatness_term = phase_flatness_metric(target_phase, out_phase, weight)\n    uniformity_term = uniformity_loss(target_amp, out_amp, weight)\n    core_uniformity_term = core_uniformity_loss(target_amp, out_amp, weight, core_region_threshold)\n    core_phase_term = core_phase_loss(target_phase, out_phase, target_amp, weight, core_region_threshold)\n    core_flatness_term = core_phase_flatness_metric(target_phase, out_phase, target_amp, weight, core_region_threshold)\n    efficiency = efficiency_metric(weight, out_amp)\n    efficiency_term = 1.0 - efficiency\n    efficiency_constraint = (\n        soft_efficiency_constraint(efficiency, efficiency_floor)\n        if efficiency_floor is not None\n        else efficiency_term\n    )\n    smoothness_term = total_variation_loss(slm_phase)\n\n    total = (\n        overlap_weight * overlap_term\n        + intensity_weight * intensity_term\n        + phase_weight * phase_term\n        + uniformity_weight * uniformity_term\n        + efficiency_weight * efficiency_constraint\n        + core_uniformity_weight * core_uniformity_term\n        + core_phase_weight * core_phase_term\n        + smoothness_weight * smoothness_term\n    )\n    metrics = {\n        \"overlap\": float(overlap.detach().cpu()),\n        \"overlap_loss\": float(overlap_term.detach().cpu()),\n        \"intensity_loss\": float(intensity_term.detach().cpu()),\n        \"phase_loss\": float(phase_term.detach().cpu()),\n        \"phase_flatness\": float(flatness_term.detach().cpu()),\n        \"uniformity_loss\": float(uniformity_term.detach().cpu()),\n        \"core_uniformity_loss\": float(core_uniformity_term.detach().cpu()),\n        \"efficiency\": float(efficiency.detach().cpu()),\n        \"efficiency_loss\": float(efficiency_term.detach().cpu()),\n        \"efficiency_constraint\": float(efficiency_constraint.detach().cpu()),\n        \"core_phase_loss\": float(core_phase_term.detach().cpu()),\n        \"core_phase_flatness\": float(core_flatness_term.detach().cpu()),\n        \"tv_loss\": float(smoothness_term.detach().cpu()),\n    }\n    return total, metrics\n",
  "ai_holography/models.py": "import math\nimport torch\nfrom torch import nn\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),\n            nn.GELU(),\n            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),\n            nn.GELU(),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.block(x)\n\n\nclass PhaseInitNet(nn.Module):\n    \"\"\"\n    Predicts an initial SLM phase from target amplitude, target phase, and weighting.\n    This is the easiest place to add \"more AI\" without giving up physical interpretability.\n    \"\"\"\n\n    def __init__(self, in_channels: int = 4, hidden: int = 32, residual_scale: float = math.pi / 2.0):\n        super().__init__()\n        self.residual_scale = residual_scale\n        self.encoder = nn.Sequential(\n            ConvBlock(in_channels, hidden),\n            nn.AvgPool2d(2),\n            ConvBlock(hidden, hidden * 2),\n            nn.AvgPool2d(2),\n            ConvBlock(hidden * 2, hidden * 4),\n        )\n        self.decoder = nn.Sequential(\n            nn.ConvTranspose2d(hidden * 4, hidden * 2, kernel_size=2, stride=2),\n            nn.GELU(),\n            ConvBlock(hidden * 2, hidden * 2),\n            nn.ConvTranspose2d(hidden * 2, hidden, kernel_size=2, stride=2),\n            nn.GELU(),\n            ConvBlock(hidden, hidden),\n            nn.Conv2d(hidden, 1, kernel_size=1),\n        )\n\n    def forward(\n        self,\n        target_amp: torch.Tensor,\n        target_phase: torch.Tensor,\n        weight: torch.Tensor,\n        base_phase: torch.Tensor,\n    ) -> torch.Tensor:\n        x = torch.stack([target_amp, target_phase / math.pi, weight, base_phase / math.pi], dim=0).unsqueeze(0)\n        residual = self.decoder(self.encoder(x)).squeeze(0).squeeze(0)\n        return base_phase + self.residual_scale * torch.tanh(residual)\n",
  "ai_holography/pipeline.py": "from __future__ import annotations\n\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nfrom .config import HolographyConfig\nfrom .calibration import load_phase_correction\nfrom .experimental_heuristics import physical_phase_guess, target_shift_pixels\nfrom .losses import composite_loss\nfrom .models import PhaseInitNet\nfrom .propagation import FourierSLM, wrap_phase\nfrom .targets import (\n    build_phase,\n    build_target,\n    build_weight,\n    gaussian_beam,\n    quadratic_phase_guess,\n    threshold_weight,\n)\nfrom .visualization import save_field_visualizations, save_linecuts\n\n\nclass AIHolographyPipeline:\n    def __init__(self, config: HolographyConfig):\n        self.cfg = config\n        self.device = torch.device(config.device)\n        self.propagator = FourierSLM(config.slm_size, config.output_size).to(self.device)\n        self.phase_net = PhaseInitNet(residual_scale=config.phase_residual_scale).to(self.device)\n        self.phase_correction = load_phase_correction(\n            config.slm_phase_correction_path,\n            size=config.slm_size,\n            device=config.device,\n        )\n        checkpoint_path = self._resolve_checkpoint()\n        self.loaded_checkpoint = checkpoint_path\n        if checkpoint_path is not None:\n            try:\n                state = torch.load(checkpoint_path, map_location=self.device)\n                self.phase_net.load_state_dict(state)\n            except Exception:\n                self.loaded_checkpoint = None\n\n    def _resolve_checkpoint(self) -> Path | None:\n        if self.cfg.checkpoint and Path(self.cfg.checkpoint).exists():\n            return Path(self.cfg.checkpoint)\n        if self.cfg.auto_load_best_checkpoint:\n            preferred: list[Path] = []\n            if self.cfg.target_type == \"target_round_top\":\n                preferred.append(Path(self.cfg.checkpoint_dir) / \"phase_init_best_round.pt\")\n            elif self.cfg.target_type == \"target_flat_top\":\n                preferred.append(Path(self.cfg.checkpoint_dir) / \"phase_init_best_flat.pt\")\n            preferred.extend(\n                [\n                    Path(self.cfg.checkpoint_dir) / \"phase_init_best_metrics.pt\",\n                    Path(self.cfg.checkpoint_dir) / \"phase_init_best.pt\",\n                ]\n            )\n            for best in preferred:\n                if best.exists():\n                    return best\n        return None\n\n    def build_problem(self) -> dict[str, torch.Tensor]:\n        shift_x_px, shift_y_px = target_shift_pixels(\n            dx_m=self.cfg.target_shift_x_m,\n            dy_m=self.cfg.target_shift_y_m,\n            wavelength_m=self.cfg.wavelength_m,\n            focal_length_m=self.cfg.focal_length_m,\n            output_shape=(self.cfg.output_size, self.cfg.output_size),\n            slm_pixel_pitch_m=self.cfg.slm_pixel_pitch_m,\n            magnification=self.cfg.magnification,\n        )\n        center = (\n            self.cfg.target_center_y if self.cfg.target_center_y is not None else self.cfg.output_size / 2.0 + shift_y_px,\n            self.cfg.target_center_x if self.cfg.target_center_x is not None else self.cfg.output_size / 2.0 + shift_x_px,\n        )\n        beam = gaussian_beam(self.cfg.slm_size, self.cfg.beam_sigma_x, self.cfg.beam_sigma_y, self.cfg.device)\n        target_kwargs = {}\n        if self.cfg.target_type == \"target_line_flat_top\":\n            target_kwargs[\"sigma_x\"] = self.cfg.line_flat_top_sigma_x\n            target_kwargs[\"sigma_y\"] = self.cfg.line_flat_top_sigma_y\n        target_amp = build_target(\n            target_type=self.cfg.target_type,\n            size=self.cfg.output_size,\n            center=center,\n            sigma=self.cfg.target_sigma,\n            charge=self.cfg.vortex_charge,\n            device=self.cfg.device,\n            **target_kwargs,\n        )\n        target_phase = build_phase(\n            phase_type=self.cfg.phase_type,\n            size=self.cfg.output_size,\n            center=center,\n            device=self.cfg.device,\n        )\n        weight_kwargs = {}\n        if self.cfg.weight_type == \"gaussian_top_line\":\n            weight_kwargs[\"sigma_x\"] = self.cfg.line_flat_top_sigma_x\n            weight_kwargs[\"sigma_y\"] = self.cfg.line_flat_top_sigma_y\n        roi = build_weight(\n            weight_type=self.cfg.weight_type,\n            size=self.cfg.output_size,\n            center=center,\n            diameter=self.cfg.roi_diameter,\n            softness=self.cfg.roi_softness,\n            device=self.cfg.device,\n            **weight_kwargs,\n        )\n        weight = threshold_weight(roi, fraction=self.cfg.weight_threshold)\n        target_amp = target_amp * weight\n        target_amp = target_amp * torch.sqrt(beam.square().sum() / target_amp.square().sum().clamp_min(1e-9))\n        return {\n            \"beam\": beam,\n            \"target_amp\": target_amp,\n            \"target_phase\": target_phase,\n            \"weight\": weight,\n        }\n\n    def predict_initial_phase(self, target_amp: torch.Tensor, target_phase: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:\n        heuristic = physical_phase_guess(\n            slm_shape=(self.cfg.slm_size, self.cfg.slm_size),\n            dx_m=self.cfg.target_shift_x_m,\n            dy_m=self.cfg.target_shift_y_m,\n            wavelength_m=self.cfg.wavelength_m,\n            focal_length_m=self.cfg.focal_length_m,\n            slm_pixel_pitch_m=self.cfg.slm_pixel_pitch_m,\n            magnification=self.cfg.magnification,\n            aspect=self.cfg.init_aspect,\n            curvature=self.cfg.init_curvature,\n            cone=self.cfg.init_cone,\n            device=self.cfg.device,\n        )\n        heuristic = heuristic + quadratic_phase_guess(\n            self.cfg.slm_size,\n            tilt=self.cfg.init_tilt,\n            aspect=self.cfg.init_aspect,\n            curvature=self.cfg.init_curvature,\n            angle=self.cfg.init_angle,\n            cone=self.cfg.init_cone,\n            device=self.cfg.device,\n        )\n        target_amp_small = self._resize_2d(target_amp, self.cfg.slm_size)\n        target_phase_small = self._resize_2d(target_phase, self.cfg.slm_size)\n        weight_small = self._resize_2d(weight, self.cfg.slm_size)\n        with torch.no_grad():\n            predicted = self.phase_net(target_amp_small, target_phase_small, weight_small, heuristic)\n        if predicted.shape != heuristic.shape:\n            predicted = F.interpolate(\n                predicted.unsqueeze(0).unsqueeze(0),\n                size=(self.cfg.slm_size, self.cfg.slm_size),\n                mode=\"bilinear\",\n                align_corners=False,\n            ).squeeze(0).squeeze(0)\n        return wrap_phase(predicted + self.phase_correction)\n\n    def _scheduled_weights(self, progress: float) -> dict[str, float]:\n        ramp_start = self.cfg.schedule_ramp_start\n        ramp_end = self.cfg.schedule_ramp_end\n        if progress <= ramp_start:\n            ramp = 0.0\n        elif progress >= ramp_end:\n            ramp = 1.0\n        else:\n            ramp = (progress - ramp_start) / max(1e-9, ramp_end - ramp_start)\n        return {\n            \"overlap_weight\": self.cfg.overlap_weight,\n            \"intensity_weight\": self.cfg.intensity_weight,\n            \"phase_weight\": self.cfg.phase_weight * (0.4 + 0.6 * ramp),\n            \"uniformity_weight\": self.cfg.uniformity_weight * ramp,\n            \"efficiency_weight\": self.cfg.efficiency_weight * ramp,\n            \"core_uniformity_weight\": self.cfg.core_uniformity_weight * ramp,\n            \"core_phase_weight\": self.cfg.core_phase_weight * ramp,\n            \"core_region_threshold\": self.cfg.core_region_threshold,\n            \"smoothness_weight\": self.cfg.smoothness_weight,\n        }\n\n    def _resize_2d(self, field: torch.Tensor, size: int) -> torch.Tensor:\n        return F.interpolate(\n            field.unsqueeze(0).unsqueeze(0),\n            size=(size, size),\n            mode=\"bilinear\",\n            align_corners=False,\n        ).squeeze(0).squeeze(0)\n\n    def _resize_phase(self, phase: torch.Tensor, size: int) -> torch.Tensor:\n        resized = self._resize_2d(phase, size)\n        return wrap_phase(resized)\n\n    def _run_adam_stage(\n        self,\n        beam: torch.Tensor,\n        target_amp: torch.Tensor,\n        target_phase: torch.Tensor,\n        weight: torch.Tensor,\n        init_phase: torch.Tensor,\n        size: int,\n        num_steps: int,\n    ) -> tuple[torch.Tensor, dict[str, float], torch.Tensor, torch.Tensor]:\n        phase = torch.nn.Parameter(self._resize_phase(init_phase, size))\n        stage_beam = beam if beam.shape == (size, size) else self._resize_2d(beam, size)\n        if size == self.cfg.slm_size:\n            stage_output_size = self.cfg.output_size\n            propagator = self.propagator\n        else:\n            stage_output_size = 2 * size\n            propagator = FourierSLM(size, stage_output_size).to(self.device)\n        stage_target_amp = target_amp if target_amp.shape == (stage_output_size, stage_output_size) else self._resize_2d(target_amp, stage_output_size)\n        stage_target_phase = target_phase if target_phase.shape == (stage_output_size, stage_output_size) else self._resize_2d(target_phase, stage_output_size)\n        stage_weight = weight if weight.shape == (stage_output_size, stage_output_size) else self._resize_2d(weight, stage_output_size)\n\n        optimizer = torch.optim.AdamW([phase], lr=self.cfg.learning_rate)\n        last_metrics: dict[str, float] = {}\n        best_loss = float(\"inf\")\n        best_phase = phase.detach().clone()\n        stale_steps = 0\n\n        for step in range(num_steps):\n            optimizer.zero_grad(set_to_none=True)\n            out_amp, out_phase = propagator(stage_beam, phase)\n            weights = self._scheduled_weights((step + 1) / max(num_steps, 1))\n            loss, last_metrics = composite_loss(\n                target_amp=stage_target_amp,\n                target_phase=stage_target_phase,\n                out_amp=out_amp,\n                out_phase=out_phase,\n                weight=stage_weight,\n                slm_phase=phase,\n                efficiency_floor=self.cfg.efficiency_floor,\n                **weights,\n            )\n            loss.backward()\n            optimizer.step()\n            if self.cfg.adaptive_weighting:\n                with torch.no_grad():\n                    target_i = stage_target_amp.square().clamp_min(1e-9)\n                    out_i = out_amp.square().clamp_min(1e-9)\n                    ratio = torch.pow(target_i / out_i, self.cfg.adaptive_weighting_alpha)\n                    ratio = torch.clamp(\n                        ratio,\n                        min=self.cfg.adaptive_weight_clip_min,\n                        max=self.cfg.adaptive_weight_clip_max,\n                    )\n                    stage_weight = stage_weight * ratio\n                    stage_weight = stage_weight / stage_weight.max().clamp_min(1e-9)\n            current_loss = float(loss.detach().cpu())\n            if current_loss + self.cfg.early_stop_min_delta < best_loss:\n                best_loss = current_loss\n                best_phase = phase.detach().clone()\n                stale_steps = 0\n            else:\n                stale_steps += 1\n            if stale_steps >= self.cfg.early_stop_patience:\n                break\n\n        final_phase = wrap_phase(best_phase)\n        out_amp, out_phase = propagator(stage_beam, final_phase)\n        _, last_metrics = composite_loss(\n            target_amp=stage_target_amp,\n            target_phase=stage_target_phase,\n            out_amp=out_amp,\n            out_phase=out_phase,\n            weight=stage_weight,\n            slm_phase=final_phase,\n            efficiency_floor=self.cfg.efficiency_floor,\n            **self._scheduled_weights(1.0),\n        )\n        return final_phase, last_metrics, out_amp.detach(), out_phase.detach()\n\n    def refine(self, beam: torch.Tensor, target_amp: torch.Tensor, target_phase: torch.Tensor, weight: torch.Tensor, init_phase: torch.Tensor) -> tuple[torch.Tensor, dict[str, float], torch.Tensor, torch.Tensor]:\n        phase = init_phase.clone()\n        last_metrics: dict[str, float] = {}\n        out_amp = out_phase = None\n\n        for level, steps in zip(self.cfg.multiscale_levels, self.cfg.stage_iterations):\n            phase, last_metrics, out_amp, out_phase = self._run_adam_stage(\n                beam=beam,\n                target_amp=target_amp,\n                target_phase=target_phase,\n                weight=weight,\n                init_phase=phase,\n                size=level,\n                num_steps=steps,\n            )\n\n        if out_amp is None or out_phase is None:\n            raise RuntimeError(\"Refinement failed to produce output fields.\")\n\n        if self.cfg.refine_with_lbfgs and not (\n            self.cfg.skip_lbfgs_if_target_met and last_metrics.get(\"overlap\", 0.0) >= self.cfg.target_overlap\n        ):\n            phase = torch.nn.Parameter(phase.clone())\n            lbfgs = torch.optim.LBFGS([phase], max_iter=self.cfg.lbfgs_steps, line_search_fn=\"strong_wolfe\")\n\n            def closure() -> torch.Tensor:\n                lbfgs.zero_grad(set_to_none=True)\n                out_amp, out_phase = self.propagator(beam, phase)\n                loss, _ = composite_loss(\n                    target_amp=target_amp,\n                    target_phase=target_phase,\n                    out_amp=out_amp,\n                    out_phase=out_phase,\n                    weight=weight,\n                    slm_phase=phase,\n                    efficiency_floor=self.cfg.efficiency_floor,\n                    **self._scheduled_weights(1.0),\n                )\n                loss.backward()\n                return loss\n\n            lbfgs.step(closure)\n            out_amp, out_phase = self.propagator(beam, phase)\n            _, last_metrics = composite_loss(\n                target_amp=target_amp,\n                target_phase=target_phase,\n                out_amp=out_amp,\n                out_phase=out_phase,\n                weight=weight,\n                slm_phase=phase,\n                efficiency_floor=self.cfg.efficiency_floor,\n                **self._scheduled_weights(1.0),\n            )\n            final_phase = wrap_phase(phase.detach())\n        else:\n            final_phase = wrap_phase(phase.detach() if isinstance(phase, torch.nn.Parameter) else phase.detach())\n        out_amp, out_phase = self.propagator(beam, final_phase)\n        return final_phase, last_metrics, out_amp.detach(), out_phase.detach()\n\n    def save_outputs(\n        self,\n        phase: torch.Tensor,\n        metrics: dict[str, float],\n        target_amp: torch.Tensor,\n        target_phase: torch.Tensor,\n        out_amp: torch.Tensor,\n        out_phase: torch.Tensor,\n    ) -> None:\n        self.cfg.output_dir.mkdir(parents=True, exist_ok=True)\n        np.save(self.cfg.output_dir / \"slm_phase.npy\", phase.cpu().numpy())\n        (self.cfg.output_dir / \"metrics.json\").write_text(json.dumps(metrics, indent=2), encoding=\"utf-8\")\n        save_field_visualizations(self.cfg.output_dir, target_amp, target_phase, out_amp, out_phase, phase)\n        save_linecuts(self.cfg.output_dir, target_amp, out_amp)\n\n    def run(self) -> tuple[torch.Tensor, dict[str, float]]:\n        start = time.perf_counter()\n        problem = self.build_problem()\n        init_phase = self.predict_initial_phase(problem[\"target_amp\"], problem[\"target_phase\"], problem[\"weight\"])\n        phase, metrics, out_amp, out_phase = self.refine(\n            problem[\"beam\"],\n            problem[\"target_amp\"],\n            problem[\"target_phase\"],\n            problem[\"weight\"],\n            init_phase,\n        )\n        metrics[\"runtime_sec\"] = time.perf_counter() - start\n        metrics[\"checkpoint\"] = str(self.loaded_checkpoint) if self.loaded_checkpoint is not None else None\n        self.save_outputs(phase, metrics, problem[\"target_amp\"], problem[\"target_phase\"], out_amp, out_phase)\n        return phase, metrics\n\n\ndef main() -> None:\n    cfg = HolographyConfig()\n    pipeline = AIHolographyPipeline(cfg)\n    _, metrics = pipeline.run()\n    print(\"Saved AI holography outputs to\", cfg.output_dir)\n    print(metrics)\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "ai_holography/profiles.py": "from __future__ import annotations\n\nfrom .config import HolographyConfig\n\n\ndef apply_profile(cfg: HolographyConfig, name: str) -> HolographyConfig:\n    if name == \"balanced\":\n        return cfg\n    if name == \"uniformity\":\n        cfg.uniformity_weight = 0.4\n        cfg.efficiency_weight = 0.08\n        cfg.phase_weight = 0.18\n        cfg.polish_uniformity_weight = 0.4\n        cfg.polish_efficiency_weight = 0.08\n        cfg.polish_phase_weight = 0.2\n        return cfg\n    if name == \"efficiency\":\n        cfg.uniformity_weight = 0.1\n        cfg.efficiency_weight = 0.3\n        cfg.phase_weight = 0.12\n        cfg.polish_uniformity_weight = 0.1\n        cfg.polish_efficiency_weight = 0.35\n        cfg.polish_phase_weight = 0.12\n        return cfg\n    if name == \"phase\":\n        cfg.uniformity_weight = 0.15\n        cfg.efficiency_weight = 0.1\n        cfg.phase_weight = 0.3\n        cfg.polish_uniformity_weight = 0.15\n        cfg.polish_efficiency_weight = 0.1\n        cfg.polish_phase_weight = 0.3\n        return cfg\n    if name == \"experiment\":\n        cfg.uniformity_weight = 0.15\n        cfg.efficiency_weight = 0.25\n        cfg.phase_weight = 0.12\n        cfg.core_uniformity_weight = 0.15\n        cfg.core_phase_weight = 0.12\n        cfg.smoothness_weight = 5e-4\n        cfg.polish_uniformity_weight = 0.12\n        cfg.polish_efficiency_weight = 0.3\n        cfg.polish_phase_weight = 0.1\n        cfg.polish_smoothness_weight = 5e-4\n        cfg.target_overlap = 0.9988\n        cfg.hybrid_cg_maxiter = 25\n        cfg.polish_stage1_maxiter = 12\n        cfg.polish_stage2_maxiter = 8\n        return cfg\n    raise ValueError(f\"Unknown profile: {name}\")\n",
  "ai_holography/propagation.py": "import math\nimport torch\nimport torch.nn.functional as F\n\n\nclass FourierSLM(torch.nn.Module):\n    def __init__(self, slm_size: int, output_size: int):\n        super().__init__()\n        if output_size < slm_size:\n            raise ValueError(f\"output_size ({output_size}) must be >= slm_size ({slm_size}).\")\n        if (output_size - slm_size) % 2 != 0:\n            raise ValueError(f\"output_size - slm_size must be even for symmetric padding, got {output_size - slm_size}.\")\n        self.slm_size = slm_size\n        self.output_size = output_size\n        self.scale = 1.0 / output_size\n        self.pad = (output_size - slm_size) // 2\n\n    def forward(self, amplitude: torch.Tensor, phase: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        field = self.scale * amplitude * torch.exp(1j * phase)\n        padded = F.pad(field, (self.pad, self.pad, self.pad, self.pad))\n        output = torch.fft.ifftshift(torch.fft.fft2(torch.fft.fftshift(padded)))\n        intensity = output.abs().square()\n        phase = torch.angle(output)\n        amplitude = torch.sqrt(intensity.clamp_min(1e-12))\n        return amplitude, phase\n\n\ndef wrap_phase(phase: torch.Tensor) -> torch.Tensor:\n    return torch.remainder(phase, 2.0 * math.pi)\n\n",
  "ai_holography/references.py": "from __future__ import annotations\n\nimport re\nfrom pathlib import Path\n\nimport torch\nimport torch.nn.functional as F\n\n\nREFERENCE_RE = re.compile(\n    r\"^795_(?P<style>[a-z_]+)_d=(?P<d>-?\\d+)(?:_sx=(?P<sx>\\d+))?(?:_sy=(?P<sy>\\d+))?_dx=(?P<dx>-?\\d+)_dy=(?P<dy>-?\\d+)\\.npy$\"\n)\n\n\ndef parse_795_reference_name(path: str | Path) -> dict[str, int | str | None]:\n    name = Path(path).name\n    match = REFERENCE_RE.match(name)\n    if not match:\n        return {\"style\": None, \"d\": None, \"dx\": None, \"dy\": None}\n    return {\n        \"style\": match.group(\"style\"),\n        \"d\": int(match.group(\"d\")),\n        \"sx\": int(match.group(\"sx\")) if match.group(\"sx\") else None,\n        \"sy\": int(match.group(\"sy\")) if match.group(\"sy\") else None,\n        \"dx\": int(match.group(\"dx\")),\n        \"dy\": int(match.group(\"dy\")),\n    }\n\n\ndef load_reference_phase(path: str | Path, size: tuple[int, int], device: str) -> torch.Tensor:\n    phase = torch.from_numpy(__import__(\"numpy\").load(path)).float().to(device)\n    if phase.ndim != 2:\n        raise ValueError(f\"Reference phase must be 2D, got shape={tuple(phase.shape)}\")\n    if phase.shape != size:\n        complex_field = torch.exp(1j * phase)\n        resized_real = F.interpolate(complex_field.unsqueeze(0).unsqueeze(0).real, size=size, mode=\"bilinear\", align_corners=False).squeeze(0).squeeze(0)\n        resized_imag = F.interpolate(complex_field.unsqueeze(0).unsqueeze(0).imag, size=size, mode=\"bilinear\", align_corners=False).squeeze(0).squeeze(0)\n        phase = torch.angle(torch.complex(resized_real, resized_imag))\n    return phase\n\n\ndef find_795_reference_files(folder: str | Path) -> list[Path]:\n    folder = Path(folder)\n    if not folder.exists():\n        return []\n    return sorted(folder.glob(\"795_*.npy\"))\n\n\ndef apply_795_reference_metadata(cfg, info: dict[str, int | str | None]):\n    style = info.get(\"style\")\n    d = info.get(\"d\") or 0\n    sx = info.get(\"sx\")\n    sy = info.get(\"sy\")\n    dx = info.get(\"dx\") or 0\n    dy = info.get(\"dy\") or 0\n\n    scale = cfg.output_size / max(cfg.reference_output_size, 1)\n    scaled_d = max(2.0, float(d) * scale)\n\n    cfg.target_center_x = cfg.output_size / 2.0 + dx * scale\n    cfg.target_shift_x_m = 0.0\n    cfg.target_center_y = cfg.output_size / 2.0 + dy * scale\n    cfg.target_shift_y_m = 0.0\n    cfg.target_sigma = scaled_d\n    if sx is not None:\n        cfg.line_flat_top_sigma_x = max(1.0, float(sx) * scale)\n    if sy is not None:\n        cfg.line_flat_top_sigma_y = max(1.0, float(sy) * scale)\n\n    if style == \"gaussian\":\n        cfg.target_type = \"target_gaussian\"\n        cfg.weight_type = \"gaussian_top_round\"\n        cfg.phase_type = \"phase_flat\"\n    elif style == \"round_top\":\n        cfg.target_type = \"target_round_top\"\n        cfg.weight_type = \"gaussian_top_round\"\n        cfg.phase_type = \"phase_flat\"\n    elif style == \"flat_top\":\n        cfg.target_type = \"target_line_flat_top\"\n        cfg.weight_type = \"gaussian_top_line\"\n        cfg.phase_type = \"phase_flat\"\n        cfg.roi_diameter = max(7.0, 1.45 * scaled_d)\n        cfg.roi_softness = max(3.5, 0.38 * scaled_d)\n    return cfg\n",
          "ai_holography/runner.py": "from __future__ import annotations\n\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nfrom lin2025_holography.config import Lin2025Config\nfrom lin2025_holography.predictor import predict_hologram_phase\n\nfrom .config import HolographyConfig\nfrom .camera_loop import (\n    CameraLoopConfig,\n    camera_feedback_weight,\n    load_measured_intensity,\n    save_measured_intensity,\n    update_target_from_measurement,\n)\nfrom .hybrid import bowman_cg_refine\nfrom .losses import composite_loss, normalized_overlap\nfrom .pipeline import AIHolographyPipeline\nfrom .propagation import wrap_phase\nfrom .references import load_reference_phase, parse_795_reference_name\nfrom .visualization import save_field_visualizations, save_linecuts\n\n\nclass HybridHolographyRunner:\n    \"\"\"\n    Production-oriented runner:\n    1. neural initialization\n    2. fast multiscale differentiable refinement\n    3. short CG polish\n    4. optional warm start from the previous generated phase\n    \"\"\"\n\n    def __init__(self, config: HolographyConfig):\n        self.cfg = config\n        self.pipeline = AIHolographyPipeline(config)\n        self.previous_phase: torch.Tensor | None = None\n        self.previous_target_amp: torch.Tensor | None = None\n        self.previous_target_phase: torch.Tensor | None = None\n        self.camera_loop = CameraLoopConfig()\n        self.last_init_source: str | None = None\n\n    def _target_similarity(self, problem: dict[str, torch.Tensor]) -> float:\n        if self.previous_target_amp is None or self.previous_target_phase is None:\n            return 0.0\n\n        amp_a = problem[\"target_amp\"]\n        amp_b = self.previous_target_amp.to(amp_a.device)\n        phase_a = problem[\"target_phase\"]\n        phase_b = self.previous_target_phase.to(phase_a.device)\n\n        amp_num = torch.sum(amp_a * amp_b)\n        amp_den = torch.sqrt(torch.sum(amp_a.square()) * torch.sum(amp_b.square())).clamp_min(1e-9)\n        amp_sim = amp_num / amp_den\n\n        phase_diff = torch.atan2(torch.sin(phase_a - phase_b), torch.cos(phase_a - phase_b))\n        phase_sim = 1.0 - torch.mean(torch.abs(phase_diff)) / torch.pi\n        score = 0.7 * amp_sim + 0.3 * phase_sim\n        return float(score.detach().cpu())\n\n    def _mix_ratio(self, similarity: float) -> float:\n        if similarity < self.cfg.warm_start_similarity_threshold:\n            return 0.0\n        span = max(1e-9, 1.0 - self.cfg.warm_start_similarity_threshold)\n        norm = min(1.0, max(0.0, (similarity - self.cfg.warm_start_similarity_threshold) / span))\n        return self.cfg.warm_start_min_mix + norm * (self.cfg.warm_start_max_mix - self.cfg.warm_start_min_mix)\n\n    def _estimate_overlap(self, problem: dict[str, torch.Tensor], phase: torch.Tensor) -> float:\n        out_amp, out_phase = self.pipeline.propagator(problem[\"beam\"], phase)\n        overlap = normalized_overlap(\n            problem[\"target_amp\"],\n            problem[\"target_phase\"],\n            out_amp,\n            out_phase,\n            problem[\"weight\"],\n        )\n        return float(overlap.detach().cpu())\n\n    def _estimate_init_score(self, problem: dict[str, torch.Tensor], phase: torch.Tensor) -> float:\n        out_amp, out_phase = self.pipeline.propagator(problem[\"beam\"], phase)\n        _, metrics = composite_loss(\n            target_amp=problem[\"target_amp\"],\n            target_phase=problem[\"target_phase\"],\n            out_amp=out_amp,\n            out_phase=out_phase,\n            weight=problem[\"weight\"],\n            slm_phase=phase,\n            overlap_weight=0.0,\n            intensity_weight=0.0,\n            phase_weight=0.0,\n            uniformity_weight=0.0,\n            efficiency_weight=0.0,\n            core_uniformity_weight=0.0,\n            core_phase_weight=0.0,\n            core_region_threshold=self.cfg.core_region_threshold,\n            smoothness_weight=0.0,\n        )\n        score = (\n            self.cfg.init_score_core_uniformity_weight * metrics[\"core_uniformity_loss\"]\n            + self.cfg.init_score_core_phase_weight * metrics[\"core_phase_flatness\"]\n            + self.cfg.init_score_efficiency_weight * (1.0 - metrics[\"efficiency\"])\n            + self.cfg.init_score_uniformity_weight * metrics[\"uniformity_loss\"]\n        )\n        return float(score)\n\n    def _build_init_phase(self, problem: dict[str, torch.Tensor], use_warm_start: bool) -> tuple[torch.Tensor, float, float]:\n        sources = set(self.cfg.init_candidate_sources)\n        neural_init = self.pipeline.predict_initial_phase(\n            problem[\"target_amp\"], problem[\"target_phase\"], problem[\"weight\"]\n        )\n        candidate_phases: dict[str, torch.Tensor] = {}\n        best_phase: torch.Tensor | None = None\n        best_overlap = float(\"-inf\")\n        best_score = float(\"inf\")\n        best_mix = 0.0\n        best_source = \"neural\"\n        if \"neural\" in sources:\n            candidate_phases[\"neural\"] = neural_init\n            best_phase = neural_init\n            best_overlap = self._estimate_overlap(problem, neural_init)\n            best_score = self._estimate_init_score(problem, neural_init)\n\n        if \"reference\" in sources and self.cfg.reference_phase_path is not None and Path(self.cfg.reference_phase_path).exists():\n            ref_phase = load_reference_phase(\n                self.cfg.reference_phase_path,\n                size=(self.cfg.slm_size, self.cfg.slm_size),\n                device=self.cfg.device,\n            )\n            ref_blend = self.cfg.reference_phase_mix * ref_phase + (1.0 - self.cfg.reference_phase_mix) * neural_init\n            candidate_phases[\"reference_blend\"] = ref_blend\n            ref_overlap = self._estimate_overlap(problem, ref_blend)\n            ref_score = self._estimate_init_score(problem, ref_blend)\n            if ref_score < best_score:\n                best_phase = ref_blend\n                best_overlap = ref_overlap\n                best_score = ref_score\n                best_mix = self.cfg.reference_phase_mix\n                best_source = \"reference_blend\"\n\n        ref_phase = None\n        ref_blend = None\n        if self.cfg.reference_phase_path is not None and Path(self.cfg.reference_phase_path).exists():\n            ref_phase = load_reference_phase(\n                self.cfg.reference_phase_path,\n                size=(self.cfg.slm_size, self.cfg.slm_size),\n                device=self.cfg.device,\n            )\n            ref_blend = self.cfg.reference_phase_mix * ref_phase + (1.0 - self.cfg.reference_phase_mix) * neural_init\n\n        if \"lin2025\" in sources and self.cfg.target_type in {\"target_flat_top\", \"target_round_top\"}:\n            lin_cfg = Lin2025Config(\n                target_mode=\"flat_top\",\n                slm_size=self.cfg.slm_size,\n                device=self.cfg.device,\n                checkpoint_dir=self.cfg.lin2025_checkpoint_dir,\n            )\n            lin_phase, lin_loaded = predict_hologram_phase(\n                lin_cfg,\n                target_amp=problem[\"target_amp\"],\n                target_phase=problem[\"target_phase\"],\n                checkpoint=self.cfg.lin2025_checkpoint,\n            )\n            candidate_phases[\"lin2025\"] = lin_phase\n            lin_overlap = self._estimate_overlap(problem, lin_phase)\n            lin_score = self._estimate_init_score(problem, lin_phase)\n            if lin_score < best_score:\n                best_phase = lin_phase\n                best_overlap = lin_overlap\n                best_score = lin_score\n                best_mix = 0.0\n                best_source = \"lin2025\"\n            if self.cfg.lin2025_primary and ref_phase is not None:\n                primary_ref_mix = self.cfg.lin2025_reference_weak_mix * ref_phase + (1.0 - self.cfg.lin2025_reference_weak_mix) * lin_phase\n                candidate_phases[\"lin2025_reference_weak_blend\"] = primary_ref_mix\n                primary_ref_overlap = self._estimate_overlap(problem, primary_ref_mix)\n                primary_ref_score = self._estimate_init_score(problem, primary_ref_mix)\n                if primary_ref_score <= best_score:\n                    best_phase = primary_ref_mix\n                    best_overlap = primary_ref_overlap\n                    best_score = primary_ref_score\n                    best_mix = self.cfg.lin2025_reference_weak_mix\n                    best_source = \"lin2025_reference_weak_blend\"\n            lin_blend = self.cfg.lin2025_mix * lin_phase + (1.0 - self.cfg.lin2025_mix) * neural_init\n            candidate_phases[\"lin2025_neural_blend\"] = lin_blend\n            lin_blend_overlap = self._estimate_overlap(problem, lin_blend)\n            lin_blend_score = self._estimate_init_score(problem, lin_blend)\n            if not self.cfg.lin2025_primary and lin_blend_score < best_score:\n                best_phase = lin_blend\n                best_overlap = lin_blend_overlap\n                best_score = lin_blend_score\n                best_mix = self.cfg.lin2025_mix\n                best_source = \"lin2025_neural_blend\"\n            if ref_phase is not None:\n                ref_lin_blend = 0.5 * ref_phase + 0.5 * lin_phase\n                candidate_phases[\"reference_lin2025_blend\"] = ref_lin_blend\n                ref_lin_overlap = self._estimate_overlap(problem, ref_lin_blend)\n                ref_lin_score = self._estimate_init_score(problem, ref_lin_blend)\n                if not self.cfg.lin2025_primary and ref_lin_score < best_score:\n                    best_phase = ref_lin_blend\n                    best_overlap = ref_lin_overlap\n                    best_score = ref_lin_score\n                    best_mix = 0.5\n                    best_source = \"reference_lin2025_blend\"\n            if lin_loaded is not None:\n                self.pipeline.loaded_checkpoint = Path(lin_loaded)\n\n        if use_warm_start and \"warm_start\" in sources and self.previous_phase is not None:\n            similarity = self._target_similarity(problem)\n            if similarity >= self.cfg.warm_start_candidate_threshold:\n                prev_phase = self.previous_phase.to(neural_init.device)\n                candidate_phases[\"warm_start\"] = prev_phase\n                if self.cfg.warm_start_allow_direct:\n                    prev_overlap = self._estimate_overlap(problem, prev_phase)\n                    prev_score = self._estimate_init_score(problem, prev_phase)\n                    if prev_score < best_score:\n                        best_phase = prev_phase\n                        best_overlap = prev_overlap\n                        best_score = prev_score\n                        best_mix = 1.0\n                        best_source = \"warm_start\"\n\n                mix = self._mix_ratio(similarity)\n                if mix > 0.0:\n                    mix = max(mix, self.cfg.warm_start_external_mix_floor)\n                    for source_name, source_phase in list(candidate_phases.items()):\n                        if source_name == \"warm_start\":\n                            continue\n                        blended = mix * prev_phase + (1.0 - mix) * source_phase\n                        blend_overlap = self._estimate_overlap(problem, blended)\n                        blend_score = self._estimate_init_score(problem, blended)\n                        if blend_score < best_score:\n                            best_phase = blended\n                            best_overlap = blend_overlap\n                            best_score = blend_score\n                            best_mix = mix\n                            best_source = f\"warm_start_{source_name}_blend\"\n                self.last_init_source = best_source\n                return best_phase, best_overlap, best_mix\n\n        if best_phase is None:\n            best_phase = neural_init\n            best_overlap = self._estimate_overlap(problem, neural_init)\n            best_source = \"neural_fallback\"\n        self.last_init_source = best_source\n        return best_phase, best_overlap, best_mix\n\n    def _phase_only_correct(\n        self,\n        problem: dict[str, torch.Tensor],\n        init_phase: torch.Tensor,\n    ) -> tuple[torch.Tensor, dict[str, float]]:\n        if not self.cfg.phase_post_correction_enabled or \"lin2025\" not in self.cfg.init_candidate_sources:\n            return init_phase, {\n                \"phase_post_correction_used\": False,\n                \"phase_post_correction_steps\": 0,\n                \"phase_post_correction_delta\": 0.0,\n            }\n\n        base_phase = init_phase.to(self.pipeline.device)\n        residual = nn.Parameter(torch.zeros_like(base_phase))\n        optimizer = torch.optim.Adam([residual], lr=self.cfg.phase_post_correction_lr)\n        best_phase = base_phase.detach().clone()\n        best_loss = float(\"inf\")\n        best_metrics: dict[str, float] = {}\n\n        for _ in range(self.cfg.phase_post_correction_steps):\n            optimizer.zero_grad(set_to_none=True)\n            phase = wrap_phase(base_phase + self.cfg.phase_post_correction_max_delta * torch.tanh(residual))\n            out_amp, out_phase = self.pipeline.propagator(problem[\"beam\"], phase)\n            loss, metrics = composite_loss(\n                target_amp=problem[\"target_amp\"],\n                target_phase=problem[\"target_phase\"],\n                out_amp=out_amp,\n                out_phase=out_phase,\n                weight=problem[\"weight\"],\n                slm_phase=phase,\n                overlap_weight=0.0,\n                intensity_weight=0.0,\n                phase_weight=0.0,\n                uniformity_weight=self.cfg.phase_post_correction_uniformity_weight,\n                efficiency_weight=self.cfg.phase_post_correction_efficiency_weight,\n                core_uniformity_weight=self.cfg.phase_post_correction_core_uniformity_weight,\n                core_phase_weight=self.cfg.phase_post_correction_core_phase_weight,\n                core_region_threshold=self.cfg.core_region_threshold,\n                smoothness_weight=self.cfg.polish_smoothness_weight,\n                efficiency_floor=self.cfg.efficiency_floor,\n            )\n            loss.backward()\n            optimizer.step()\n            loss_value = float(loss.detach().cpu())\n            if loss_value < best_loss:\n                best_loss = loss_value\n                best_phase = phase.detach().clone()\n                best_metrics = metrics\n\n        delta = torch.atan2(torch.sin(best_phase - base_phase), torch.cos(best_phase - base_phase))\n        correction_metrics = {\n            \"phase_post_correction_used\": True,\n            \"phase_post_correction_steps\": self.cfg.phase_post_correction_steps,\n            \"phase_post_correction_delta\": float(delta.abs().mean().detach().cpu()),\n            \"phase_post_correction_core_uniformity_loss\": best_metrics.get(\"core_uniformity_loss\", 0.0),\n            \"phase_post_correction_core_phase_flatness\": best_metrics.get(\"core_phase_flatness\", 0.0),\n            \"phase_post_correction_efficiency\": best_metrics.get(\"efficiency\", 0.0),\n        }\n        return best_phase, correction_metrics\n\n    def run_once(\n        self,\n        use_warm_start: bool = True,\n        save_subdir: str | None = None,\n    ) -> tuple[torch.Tensor, dict[str, float]]:\n        start = time.perf_counter()\n        problem = self.pipeline.build_problem()\n        similarity = self._target_similarity(problem) if use_warm_start else 0.0\n        init_phase, init_overlap, warm_mix = self._build_init_phase(problem, use_warm_start=use_warm_start)\n        corrected_phase, correction_metrics = self._phase_only_correct(problem, init_phase)\n        corrected_overlap = self._estimate_overlap(problem, corrected_phase)\n        if self._estimate_init_score(problem, corrected_phase) <= self._estimate_init_score(problem, init_phase):\n            init_phase = corrected_phase\n            init_overlap = corrected_overlap\n\n        if init_overlap >= self.cfg.fast_init_overlap_threshold:\n            refined_phase = init_phase\n            ai_metrics = {\"overlap\": init_overlap}\n        else:\n            refined_phase, ai_metrics, _, _ = self.pipeline.refine(\n                problem[\"beam\"],\n                problem[\"target_amp\"],\n                problem[\"target_phase\"],\n                problem[\"weight\"],\n                init_phase,\n            )\n\n        if self.camera_loop.enabled:\n            measured = load_measured_intensity(\n                self.camera_loop.measured_intensity_path,\n                expected_shape=tuple(problem[\"target_amp\"].shape),\n            )\n            if measured is not None:\n                correction = camera_feedback_weight(measured, problem[\"target_amp\"].square())\n                correction = correction.to(problem[\"weight\"].device)\n                problem[\"weight\"] = problem[\"weight\"] * correction / correction.max().clamp_min(1e-9)\n                problem[\"target_amp\"] = update_target_from_measurement(\n                    previous_target_amp=problem[\"target_amp\"],\n                    measured_intensity=measured.to(problem[\"target_amp\"].device),\n                    gain=self.cfg.feedback_target_gain,\n                )\n\n        hybrid = bowman_cg_refine(\n            self.cfg,\n            problem,\n            refined_phase,\n            maxiter=self.cfg.hybrid_cg_maxiter,\n            target_overlap=self.cfg.target_overlap,\n        )\n        hybrid[\"metrics\"][\"checkpoint\"] = (\n            str(self.pipeline.loaded_checkpoint) if self.pipeline.loaded_checkpoint is not None else None\n        )\n        hybrid[\"metrics\"][\"ai_overlap_before_cg\"] = ai_metrics.get(\"overlap\")\n        hybrid[\"metrics\"][\"init_overlap\"] = init_overlap\n        hybrid[\"metrics\"][\"runtime_total_sec\"] = time.perf_counter() - start\n        hybrid[\"metrics\"][\"used_warm_start\"] = bool(use_warm_start and self.previous_phase is not None and warm_mix > 0.0)\n        hybrid[\"metrics\"][\"warm_start_similarity\"] = similarity\n        hybrid[\"metrics\"][\"warm_start_mix\"] = warm_mix\n        hybrid[\"metrics\"][\"init_source\"] = self.last_init_source\n        hybrid[\"metrics\"][\"skipped_ai_refine\"] = bool(init_overlap >= self.cfg.fast_init_overlap_threshold)\n        hybrid[\"metrics\"][\"camera_loop_enabled\"] = self.camera_loop.enabled\n        hybrid[\"metrics\"].update(correction_metrics)\n        if self.cfg.reference_phase_path is not None:\n            hybrid[\"metrics\"][\"reference_phase_path\"] = str(self.cfg.reference_phase_path)\n            hybrid[\"metrics\"][\"reference_info\"] = parse_795_reference_name(self.cfg.reference_phase_path)\n\n        output_dir = self.cfg.output_dir if save_subdir is None else self.cfg.output_dir / save_subdir\n        output_dir.mkdir(parents=True, exist_ok=True)\n        np.save(output_dir / \"slm_phase.npy\", hybrid[\"phase\"].numpy())\n        (output_dir / \"metrics.json\").write_text(json.dumps(hybrid[\"metrics\"], indent=2), encoding=\"utf-8\")\n        save_field_visualizations(\n            output_dir,\n            problem[\"target_amp\"],\n            problem[\"target_phase\"],\n            hybrid[\"out_amp\"],\n            hybrid[\"out_phase\"],\n            hybrid[\"phase\"],\n        )\n        save_linecuts(output_dir, problem[\"target_amp\"], hybrid[\"out_amp\"])\n\n        self.previous_phase = hybrid[\"phase\"].clone()\n        self.previous_target_amp = problem[\"target_amp\"].detach().cpu().clone()\n        self.previous_target_phase = problem[\"target_phase\"].detach().cpu().clone()\n        return hybrid[\"phase\"], hybrid[\"metrics\"]\n\n    def run_closed_loop(\n        self,\n        measured_intensity_path: str | None = None,\n        save_root: str = \"closed_loop\",\n        synthesize_measurement_from_first_pass: bool = True,\n    ) -> list[dict[str, float]]:\n        records: list[dict[str, float]] = []\n\n        # Round 1: open-loop generation\n        phase0, metrics0 = self.run_once(use_warm_start=False, save_subdir=f\"{save_root}_round0\")\n        records.append({\"round\": 0.0, **metrics0})\n\n        output_dir0 = self.cfg.output_dir / f\"{save_root}_round0\"\n        round0_metrics_path = output_dir0 / \"metrics.json\"\n        round0_phase_path = output_dir0 / \"slm_phase.npy\"\n        _ = phase0, round0_metrics_path, round0_phase_path\n\n        # Use provided measured image or synthesize one from the first-pass simulation.\n        measurement_file = measured_intensity_path\n        if measurement_file is None and synthesize_measurement_from_first_pass:\n            problem = self.pipeline.build_problem()\n            phase_tensor = torch.tensor(np.load(output_dir0 / \"slm_phase.npy\"), dtype=torch.float32, device=self.pipeline.device)\n            out_amp, _ = self.pipeline.propagator(problem[\"beam\"], phase_tensor)\n            measured_i = out_amp.square()\n            measurement_file = str(self.cfg.output_dir / f\"{save_root}_measured.npy\")\n            save_measured_intensity(measurement_file, measured_i)\n\n        # Round 2: camera-feedback generation\n        self.camera_loop.enabled = True\n        self.camera_loop.measured_intensity_path = measurement_file\n        phase1, metrics1 = self.run_once(use_warm_start=False, save_subdir=f\"{save_root}_round1\")\n        records.append({\"round\": 1.0, **metrics1})\n        self.camera_loop.enabled = False\n        self.camera_loop.measured_intensity_path = None\n\n        summary = {\n            \"round0\": metrics0,\n            \"round1\": metrics1,\n            \"delta\": {\n                \"overlap\": metrics1.get(\"overlap\", 0.0) - metrics0.get(\"overlap\", 0.0),\n                \"efficiency\": metrics1.get(\"efficiency\", 0.0) - metrics0.get(\"efficiency\", 0.0),\n                \"uniformity_loss\": metrics1.get(\"uniformity_loss\", 0.0) - metrics0.get(\"uniformity_loss\", 0.0),\n                \"phase_flatness\": metrics1.get(\"phase_flatness\", 0.0) - metrics0.get(\"phase_flatness\", 0.0),\n            },\n            \"measurement_file\": measurement_file,\n        }\n        summary_path = self.cfg.output_dir / f\"{save_root}_summary.json\"\n        summary_path.write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n        return records\n\n    def run_sequence(self, count: int, save_root: str = \"sequence\") -> list[dict[str, float]]:\n        metrics_list: list[dict[str, float]] = []\n        for index in range(count):\n            _, metrics = self.run_once(use_warm_start=True, save_subdir=f\"{save_root}_{index:03d}\")\n            metrics_list.append(metrics)\n        summary_path = self.cfg.output_dir / f\"{save_root}_summary.json\"\n        summary_path.write_text(json.dumps(metrics_list, indent=2), encoding=\"utf-8\")\n        return metrics_list\n",
  "ai_holography/targets.py": "import math\nimport torch\n\n\ndef _meshgrid(size: int, device: str) -> tuple[torch.Tensor, torch.Tensor]:\n    axis = torch.arange(size, dtype=torch.float32, device=device)\n    return torch.meshgrid(axis, axis, indexing=\"ij\")\n\n\ndef gaussian_beam(size: int, sigma_x: float, sigma_y: float, device: str) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    cx = cy = size / 2.0\n    beam = torch.exp(\n        -2.0 * (((x - cx) / sigma_x) ** 2 + ((y - cy) / sigma_y) ** 2)\n    )\n    return beam / beam.square().sum().sqrt().clamp_min(1e-9)\n\n\ndef laguerre_gaussian_target(\n    size: int,\n    center: tuple[float, float],\n    width: float,\n    charge: int,\n    device: str,\n) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    cy, cx = center\n    r = torch.sqrt((x - cx) ** 2 + (y - cy) ** 2)\n    radial = torch.pow((r * math.sqrt(2.0) / width).clamp_min(1e-9), abs(charge))\n    amplitude = (radial * torch.exp(-((r / width) ** 2)) * 2.0 * (r / width) ** 2) / width\n    return amplitude\n\n\ndef gaussian_target(\n    size: int,\n    center: tuple[float, float],\n    sigma_x: float,\n    sigma_y: float,\n    device: str,\n) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    cy, cx = center\n    return torch.exp(-2.0 * (((x - cx) / sigma_x) ** 2 + ((y - cy) / sigma_y) ** 2))\n\n\ndef round_top_target(\n    size: int,\n    center: tuple[float, float],\n    diameter: float,\n    edge_softness: float,\n    device: str,\n) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    cy, cx = center\n    radius = torch.sqrt((x - cx) ** 2 + (y - cy) ** 2)\n    transition = (radius - diameter / 2.0) / max(edge_softness, 1e-6)\n    return torch.sigmoid(-transition)\n\n\ndef flat_top_target(\n    size: int,\n    center: tuple[float, float],\n    diameter: float,\n    order: float,\n    edge_softness: float,\n    device: str,\n) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    cy, cx = center\n    radius = torch.sqrt((x - cx) ** 2 + (y - cy) ** 2)\n    norm_r = radius / max(diameter / 2.0, 1e-6)\n    soft_transition = torch.sigmoid(-(norm_r - 1.0) / max(edge_softness, 1e-6))\n    super_gaussian = torch.exp(-torch.pow(norm_r, order))\n    return soft_transition * super_gaussian\n\n\ndef line_flat_top_target(\n    size: int,\n    center: tuple[float, float],\n    width_x: float,\n    sigma_x: float,\n    sigma_y: float,\n    device: str,\n) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    cy, cx = center\n    dx = x - cx\n    dy = y - cy\n    fx = 0.5 * (torch.abs(dx - width_x / 2.0) + torch.abs(dx + width_x / 2.0) - width_x)\n    return torch.exp(-(fx.square()) / max(sigma_x, 1e-6) ** 2) * torch.exp(-(dy.square()) / max(sigma_y, 1e-6) ** 2)\n\n\ndef vortex_phase(size: int, center: tuple[float, float], device: str) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    cy, cx = center\n    phase = torch.atan2(x - cx, y - cy)\n    return torch.remainder(phase + math.pi, 2.0 * math.pi) - math.pi\n\n\ndef circular_weight(size: int, center: tuple[float, float], diameter: float, softness: float, device: str) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    cy, cx = center\n    radius = torch.sqrt((x - cx) ** 2 + (y - cy) ** 2)\n    shell = 0.5 * (torch.abs(radius - diameter / 2.0) + torch.abs(radius + diameter / 2.0) - diameter)\n    mask = torch.exp(-((shell / softness) ** 2))\n    return torch.where(mask >= 1e-5, mask, torch.zeros_like(mask))\n\n\ndef line_weight(size: int, center: tuple[float, float], width_x: float, sigma_x: float, sigma_y: float, device: str) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    cy, cx = center\n    dx = x - cx\n    dy = y - cy\n    fx = 0.5 * (torch.abs(dx - width_x / 2.0) + torch.abs(dx + width_x / 2.0) - width_x)\n    wx = torch.exp(-(fx.square()) / max(sigma_x * 1.35, 1e-6) ** 2)\n    wy = torch.exp(-(dy.square()) / max(sigma_y * 1.35, 1e-6) ** 2)\n    mask = wx * wy\n    return torch.where(mask >= 1e-5, mask, torch.zeros_like(mask))\n\n\ndef threshold_weight(mask: torch.Tensor, fraction: float, fill: float = 0.0) -> torch.Tensor:\n    threshold = fraction * mask.max().clamp_min(1e-9)\n    return torch.where(mask.abs() < threshold, torch.full_like(mask, fill), torch.ones_like(mask))\n\n\ndef quadratic_phase_guess(size: int, tilt: float, aspect: float, curvature: float, angle: float, cone: float, device: str) -> torch.Tensor:\n    y, x = _meshgrid(size, device)\n    x = x - size / 2.0\n    y = y - size / 2.0\n    linear = tilt * (x * torch.cos(torch.tensor(angle, device=device)) + y * torch.sin(torch.tensor(angle, device=device)))\n    quad = 3.0 * curvature * (aspect * x.square() + (1.0 - aspect) * y.square())\n    radial = cone * torch.sqrt(x.square() + y.square())\n    return linear + quad + radial\n\n\ndef flat_phase(size: int, device: str) -> torch.Tensor:\n    return torch.zeros((size, size), dtype=torch.float32, device=device)\n\n\ndef build_target(target_type: str, size: int, center: tuple[float, float], sigma: float, charge: int, device: str, **kwargs) -> torch.Tensor:\n    if target_type == \"target_lg\":\n        return laguerre_gaussian_target(size=size, center=center, width=sigma, charge=charge, device=device)\n    if target_type == \"target_gaussian\":\n        return gaussian_target(size=size, center=center, sigma_x=sigma, sigma_y=sigma, device=device)\n    if target_type == \"target_round_top\":\n        return round_top_target(size=size, center=center, diameter=sigma, edge_softness=max(2.0, sigma * 0.08), device=device)\n    if target_type == \"target_flat_top\":\n        return flat_top_target(\n            size=size,\n            center=center,\n            diameter=sigma,\n            order=8.0,\n            edge_softness=max(0.08, 0.12),\n            device=device,\n        )\n    if target_type == \"target_line_flat_top\":\n        return line_flat_top_target(\n            size=size,\n            center=center,\n            width_x=sigma,\n            sigma_x=kwargs.get(\"sigma_x\", max(2.0, sigma * 0.20)),\n            sigma_y=kwargs.get(\"sigma_y\", max(2.0, sigma * 0.20)),\n            device=device,\n        )\n    raise ValueError(f\"Unsupported target_type: {target_type}\")\n\n\ndef build_phase(phase_type: str, size: int, center: tuple[float, float], device: str) -> torch.Tensor:\n    if phase_type == \"phase_spinning_continuous\":\n        return vortex_phase(size=size, center=center, device=device)\n    if phase_type == \"phase_flat\":\n        return flat_phase(size=size, device=device)\n    raise ValueError(f\"Unsupported phase_type: {phase_type}\")\n\n\ndef build_weight(weight_type: str, size: int, center: tuple[float, float], diameter: float, softness: float, device: str, **kwargs) -> torch.Tensor:\n    if weight_type == \"gaussian_top_round\":\n        return circular_weight(size=size, center=center, diameter=diameter, softness=softness, device=device)\n    if weight_type == \"gaussian_top_line\":\n        return line_weight(size=size, center=center, width_x=diameter, sigma_x=kwargs.get(\"sigma_x\", max(2.0, diameter * 0.20)), sigma_y=kwargs.get(\"sigma_y\", max(2.0, diameter * 0.20)), device=device)\n    raise ValueError(f\"Unsupported weight_type: {weight_type}\")\n",
          "ai_holography/visualization.py": "from pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\n\n\ndef _to_numpy(x: torch.Tensor | np.ndarray) -> np.ndarray:\n    if isinstance(x, torch.Tensor):\n        return x.detach().cpu().numpy()\n    return x\n\n\ndef save_field_visualizations(\n    output_dir: Path,\n    target_amp: torch.Tensor,\n    target_phase: torch.Tensor,\n    out_amp: torch.Tensor,\n    out_phase: torch.Tensor,\n    slm_phase: torch.Tensor,\n) -> None:\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    ta = _to_numpy(target_amp)\n    tp = _to_numpy(target_phase)\n    oa = _to_numpy(out_amp)\n    op = _to_numpy(out_phase)\n    sp = _to_numpy(slm_phase)\n\n    fig, axes = plt.subplots(2, 3, figsize=(12, 8))\n    items = [\n        (ta**2, \"Target Intensity\", \"viridis\"),\n        (tp, \"Target Phase\", \"twilight\"),\n        (oa**2, \"Output Intensity\", \"viridis\"),\n        (op, \"Output Phase\", \"twilight\"),\n        (sp, \"SLM Phase\", \"twilight\"),\n        ((ta**2) - (oa**2), \"Intensity Error\", \"coolwarm\"),\n    ]\n    for ax, (img, title, cmap) in zip(axes.flat, items):\n        im = ax.imshow(img, cmap=cmap)\n        ax.set_title(title)\n        ax.set_xticks([])\n        ax.set_yticks([])\n        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    fig.tight_layout()\n    fig.savefig(output_dir / \"summary.png\", dpi=180)\n    plt.close(fig)\n\n\ndef save_linecuts(\n    output_dir: Path,\n    target_amp: torch.Tensor,\n    out_amp: torch.Tensor,\n) -> None:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    ta = _to_numpy(target_amp**2)\n    oa = _to_numpy(out_amp**2)\n\n    cy = ta.shape[0] // 2\n    cx = ta.shape[1] // 2\n\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n    axes[0].plot(ta[cy, :], label=\"target\")\n    axes[0].plot(oa[cy, :], label=\"output\")\n    axes[0].set_title(\"Horizontal Linecut\")\n    axes[0].legend()\n\n    axes[1].plot(ta[:, cx], label=\"target\")\n    axes[1].plot(oa[:, cx], label=\"output\")\n    axes[1].set_title(\"Vertical Linecut\")\n    axes[1].legend()\n\n    fig.tight_layout()\n    fig.savefig(output_dir / \"linecuts.png\", dpi=180)\n    plt.close(fig)\n\n",
  "lin2025_holography/__init__.py": "\"\"\"Lin 2025-inspired hologram generation pipeline.\"\"\"\n\n",
  "lin2025_holography/config.py": "from dataclasses import dataclass, field\nfrom datetime import datetime\nfrom pathlib import Path\n\n\n@dataclass\nclass Lin2025Config:\n    target_mode: str = \"trap_array\"\n    slm_size: int = 128\n    oversample_factor: int = 4\n    num_traps: int = 64\n    min_spacing_px: float = 6.0\n    trap_sigma_px: float = 1.8\n    beam_sigma_px: float = 28.0\n    reference_dir: Path = Path(\"ftl_gen\")\n    reference_output_size: int = 6876\n    target_sigma_px: float = 18.0\n    roi_softness_px: float = 2.0\n    weight_threshold: float = 1e-4\n    wgs_iterations: int = 20\n    train_samples: int = 192\n    val_samples: int = 24\n    epochs: int = 8\n    batch_size: int = 4\n    learning_rate: float = 3e-4\n    grad_clip_norm: float = 1.0\n    skip_nonfinite_batches: bool = True\n    scheduler_patience: int = 1\n    scheduler_factor: float = 0.5\n    scheduler_min_lr: float = 1e-5\n    device: str = \"cpu\"\n    input_channels: int = 4\n    hidden_channels: int = 48\n    num_residual_blocks: int = 4\n    flat_overlap_weight: float = 0.1\n    flat_uniformity_weight: float = 0.25\n    flat_core_uniformity_weight: float = 1.15\n    flat_efficiency_weight: float = 0.35\n    flat_core_phase_weight: float = 1.8\n    flat_phase_weight: float = 0.1\n    flat_core_threshold: float = 0.78\n    efficiency_floor: float = 0.15\n    teacher_sources: tuple[str, ...] = (\"795\", \"hybrid\")\n    teacher_weight_795: float = 0.1\n    teacher_weight_hybrid: float = 0.9\n    teacher_weight_795_round: float = 0.4\n    teacher_weight_hybrid_round: float = 0.6\n    teacher_weight_795_flat: float = 0.15\n    teacher_weight_hybrid_flat: float = 0.85\n    hybrid_teacher_phase_gate: float = 0.0018\n    hybrid_teacher_phase_gate_round: float = 0.0025\n    hybrid_teacher_phase_gate_flat: float = 0.0022\n    hybrid_teacher_top_k: int = 6\n    hybrid_teacher_top_k_round: int = 6\n    hybrid_teacher_top_k_flat: int = 4\n    hybrid_teacher_score_uniformity_weight: float = 0.15\n    hybrid_teacher_score_core_uniformity_weight: float = 1.2\n    hybrid_teacher_score_efficiency_weight: float = 0.4\n    hybrid_teacher_score_core_phase_weight: float = 1.3\n    hybrid_teacher_score_uniformity_weight_round: float = 0.15\n    hybrid_teacher_score_core_uniformity_weight_round: float = 1.2\n    hybrid_teacher_score_efficiency_weight_round: float = 0.4\n    hybrid_teacher_score_core_phase_weight_round: float = 1.3\n    hybrid_teacher_score_uniformity_weight_flat: float = 0.05\n    hybrid_teacher_score_core_uniformity_weight_flat: float = 0.7\n    hybrid_teacher_score_efficiency_weight_flat: float = 0.15\n    hybrid_teacher_score_core_phase_weight_flat: float = 2.2\n    hybrid_teacher_round_path: Path | None = None\n    hybrid_teacher_flat_path: Path | None = None\n    hybrid_teacher_search_root: Path = Path(\"ai_holography\") / \"runs\"\n    hybrid_init_overlap_weight: float = 0.1\n    hybrid_init_uniformity_weight: float = 0.2\n    hybrid_init_core_uniformity_weight: float = 1.0\n    hybrid_init_efficiency_weight: float = 0.5\n    hybrid_init_core_phase_weight: float = 1.2\n    hologram_phase_imitation_weight: float = 0.2\n    hologram_core_phase_imitation_weight: float = 1.1\n    hologram_roi_phase_imitation_weight: float = 0.35\n    metric_score_uniformity_weight: float = 0.15\n    metric_score_core_uniformity_weight: float = 1.2\n    metric_score_efficiency_weight: float = 0.4\n    metric_score_core_phase_weight: float = 1.3\n    run_name: str = field(default_factory=lambda: datetime.now().strftime(\"%Y%m%d_%H%M%S\"))\n    run_dir: Path = field(default_factory=lambda: Path(\"lin2025_holography\") / \"runs\")\n    checkpoint_dir: Path = Path(\"lin2025_holography\") / \"checkpoints\"\n    output_dir: Path | None = None\n\n    def __post_init__(self) -> None:\n        self.run_dir = Path(self.run_dir) / self.run_name\n        if self.output_dir is None:\n            self.output_dir = self.run_dir / \"outputs\"\n        else:\n            self.output_dir = Path(self.output_dir)\n\n    @property\n    def oversampled_size(self) -> int:\n        return self.slm_size * self.oversample_factor\n",
  "lin2025_holography/dataset.py": "from __future__ import annotations\n\nimport copy\nimport json\nimport random\nfrom pathlib import Path\n\nimport torch\n\nfrom ai_holography.losses import target_core_weight\nfrom ai_holography.references import apply_795_reference_metadata, find_795_reference_files, load_reference_phase, parse_795_reference_name\nfrom ai_holography.targets import build_phase, build_target, build_weight, threshold_weight\n\nfrom .config import Lin2025Config\nfrom .encoding import encode_inputs\nfrom .wgs import hologram_to_position_labels, weighted_gs_hologram\n\n\nclass OnTheFlyWGSDataset:\n    def __init__(self, cfg: Lin2025Config, num_samples: int, seed: int = 42):\n        self.cfg = cfg\n        self.num_samples = num_samples\n        self.rng = random.Random(seed)\n        self.device = torch.device(cfg.device)\n\n    def __len__(self) -> int:\n        return self.num_samples\n\n    def _sample_coords(self) -> torch.Tensor:\n        coords: list[tuple[float, float]] = []\n        attempts = 0\n        margin = self.cfg.slm_size * 0.12\n        while len(coords) < self.cfg.num_traps and attempts < self.cfg.num_traps * 200:\n            attempts += 1\n            x = self.rng.uniform(margin, self.cfg.slm_size - margin)\n            y = self.rng.uniform(margin, self.cfg.slm_size - margin)\n            if all((x - px) ** 2 + (y - py) ** 2 >= self.cfg.min_spacing_px**2 for px, py in coords):\n                coords.append((x, y))\n        if len(coords) < self.cfg.num_traps:\n            raise RuntimeError(\"Failed to sample enough non-colliding trap coordinates.\")\n        return torch.tensor(coords, dtype=torch.float32, device=self.device)\n\n    def _sample_phases(self) -> torch.Tensor:\n        return (2.0 * torch.pi * torch.rand((self.cfg.num_traps,), device=self.device) - torch.pi).float()\n\n    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:\n        del idx\n        coords = self._sample_coords()\n        phases = self._sample_phases()\n        a_input, phi_input = encode_inputs(coords, phases, size=self.cfg.slm_size, device=self.device)\n        hologram = weighted_gs_hologram(\n            coords=coords,\n            phases=phases,\n            slm_size=self.cfg.slm_size,\n            oversampled_size=self.cfg.oversampled_size,\n            trap_sigma=self.cfg.trap_sigma_px,\n            beam_sigma=self.cfg.beam_sigma_px,\n            iterations=self.cfg.wgs_iterations,\n            device=self.device,\n        )\n        a_label, phi_label = hologram_to_position_labels(hologram, label_size=self.cfg.slm_size)\n        return {\n            \"a_input\": a_input,\n            \"phi_input\": phi_input,\n            \"a_label\": a_label,\n            \"phi_label\": phi_label,\n            \"coords\": coords,\n            \"trap_phases\": phases,\n            \"mode\": \"trap_array\",\n        }\n\n\nclass FlatTopReferenceDataset:\n    \"\"\"\n    Lin-style supervised dataset adapted to flat-top holography:\n    - input is the target amplitude/phase image in the position domain\n    - label supervision comes from 795-style phase-only holograms\n    \"\"\"\n\n    def __init__(self, cfg: Lin2025Config, num_samples: int, seed: int = 42):\n        self.cfg = cfg\n        self.num_samples = num_samples\n        self.rng = random.Random(seed)\n        self.device = torch.device(cfg.device)\n        refs = find_795_reference_files(Path(cfg.reference_dir))\n        self.refs = []\n        for ref in refs:\n            info = parse_795_reference_name(ref)\n            if info.get(\"style\") == \"flat_top\" and info.get(\"sx\") is not None and info.get(\"sy\") is not None:\n                self.refs.append(ref)\n        if not self.refs:\n            raise FileNotFoundError(f\"No line-flat 795 references with sx/sy metadata found in {cfg.reference_dir}\")\n        self.hybrid_teachers = self._discover_hybrid_teachers()\n\n    def _discover_hybrid_teachers(self) -> dict[str, list[Path]]:\n        teachers: dict[str, list[Path]] = {}\n        explicit = {\n            \"round_top\": self.cfg.hybrid_teacher_round_path,\n            \"flat_top\": self.cfg.hybrid_teacher_flat_path,\n        }\n        for style, path in explicit.items():\n            if path is not None and Path(path).exists():\n                teachers[style] = [Path(path)]\n\n        search_root = Path(self.cfg.hybrid_teacher_search_root)\n        if search_root.exists():\n            if \"round_top\" not in teachers:\n                selected = self._select_best_hybrid_teachers(search_root, \"round_top\")\n                if selected:\n                    teachers[\"round_top\"] = selected\n            if \"flat_top\" not in teachers:\n                selected = self._select_best_hybrid_teachers(search_root, \"flat_top\")\n                if selected:\n                    teachers[\"flat_top\"] = selected\n        return teachers\n\n    def _teacher_score_weights(self, style: str) -> tuple[float, float, float, float]:\n        if style == \"flat_top\":\n            return (\n                self.cfg.hybrid_teacher_score_uniformity_weight_flat,\n                self.cfg.hybrid_teacher_score_core_uniformity_weight_flat,\n                self.cfg.hybrid_teacher_score_efficiency_weight_flat,\n                self.cfg.hybrid_teacher_score_core_phase_weight_flat,\n            )\n        return (\n            self.cfg.hybrid_teacher_score_uniformity_weight_round,\n            self.cfg.hybrid_teacher_score_core_uniformity_weight_round,\n            self.cfg.hybrid_teacher_score_efficiency_weight_round,\n            self.cfg.hybrid_teacher_score_core_phase_weight_round,\n        )\n\n    def _teacher_phase_gate(self, style: str) -> float:\n        if style == \"flat_top\":\n            return self.cfg.hybrid_teacher_phase_gate_flat\n        return self.cfg.hybrid_teacher_phase_gate_round\n\n    def _teacher_top_k(self, style: str) -> int:\n        if style == \"flat_top\":\n            return max(1, self.cfg.hybrid_teacher_top_k_flat)\n        return max(1, self.cfg.hybrid_teacher_top_k_round)\n\n    def _teacher_score(self, style: str, metrics: dict[str, float]) -> float:\n        uniformity_weight, core_uniformity_weight, efficiency_weight, core_phase_weight = self._teacher_score_weights(style)\n        return (\n            core_uniformity_weight * float(metrics.get(\"core_uniformity_loss\", float(\"inf\")))\n            + core_phase_weight * float(metrics.get(\"core_phase_flatness\", float(\"inf\")))\n            + efficiency_weight * (1.0 - float(metrics.get(\"efficiency\", 0.0)))\n            + uniformity_weight * float(metrics.get(\"uniformity_loss\", float(\"inf\")))\n        )\n\n    def _teacher_patterns(self, style: str) -> list[str]:\n        if style == \"flat_top\":\n            return [\n                \"**/outputs_flat_top/slm_phase.npy\",\n                \"**/outputs/flat_*/slm_phase.npy\",\n            ]\n        return [\n            \"**/outputs_round_top/slm_phase.npy\",\n            \"**/outputs/round_*/slm_phase.npy\",\n        ]\n\n    def _select_best_hybrid_teachers(self, search_root: Path, style: str) -> list[Path]:\n        candidates: list[tuple[float, float, Path]] = []\n        gated: list[tuple[float, float, Path]] = []\n        seen: set[Path] = set()\n        for pattern in self._teacher_patterns(style):\n            for phase_path in search_root.glob(pattern):\n                phase_path = phase_path.resolve()\n                if phase_path in seen:\n                    continue\n                seen.add(phase_path)\n                metrics_path = phase_path.with_name(\"metrics.json\")\n                if not metrics_path.exists():\n                    continue\n                try:\n                    metrics = json.loads(metrics_path.read_text(encoding=\"utf-8\"))\n                except Exception:\n                    continue\n                if style == \"flat_top\":\n                    ref_info = metrics.get(\"reference_info\", {}) if isinstance(metrics, dict) else {}\n                    ref_path = str(metrics.get(\"reference_phase_path\", \"\")) if isinstance(metrics, dict) else \"\"\n                    sx = ref_info.get(\"sx\") if isinstance(ref_info, dict) else None\n                    sy = ref_info.get(\"sy\") if isinstance(ref_info, dict) else None\n                    if (sx is None or sy is None) and (\"_sx=\" not in ref_path or \"_sy=\" not in ref_path):\n                        continue\n                core_phase_flatness = float(metrics.get(\"core_phase_flatness\", float(\"inf\")))\n                core_uniformity_loss = float(metrics.get(\"core_uniformity_loss\", float(\"inf\")))\n                efficiency = float(metrics.get(\"efficiency\", 0.0))\n                score = self._teacher_score(style, metrics)\n                entry = (score, core_phase_flatness, phase_path)\n                candidates.append(entry)\n                if core_phase_flatness <= self._teacher_phase_gate(style):\n                    gated.append(entry)\n        pool = gated if gated else candidates\n        pool.sort(key=lambda item: (item[0], item[1]))\n        selected = [path for _, _, path in pool[: self._teacher_top_k(style)]]\n        return selected\n\n    def __len__(self) -> int:\n        return self.num_samples\n\n    def _source_weights_for_style(self, style: str) -> dict[str, float]:\n        if style == \"flat_top\":\n            return {\n                \"795\": self.cfg.teacher_weight_795_flat,\n                \"hybrid\": self.cfg.teacher_weight_hybrid_flat,\n            }\n        return {\n            \"795\": self.cfg.teacher_weight_795_round,\n            \"hybrid\": self.cfg.teacher_weight_hybrid_round,\n        }\n\n    def _choose_teacher_source(self, style: str) -> str:\n        weights = self._source_weights_for_style(style)\n        hybrid_available = bool(self.hybrid_teachers.get(style))\n        pool: list[str] = []\n        if \"795\" in self.cfg.teacher_sources and weights[\"795\"] > 0:\n            pool.extend([\"795\"] * max(1, int(round(weights[\"795\"] * 100))))\n        if \"hybrid\" in self.cfg.teacher_sources and hybrid_available and weights[\"hybrid\"] > 0:\n            pool.extend([\"hybrid\"] * max(1, int(round(weights[\"hybrid\"] * 100))))\n        if not pool:\n            return \"795\"\n        return self.rng.choice(pool)\n\n    def _choose_hybrid_teacher_path(self, style: str) -> Path | None:\n        candidates = self.hybrid_teachers.get(style, [])\n        if not candidates:\n            return None\n        if len(candidates) == 1:\n            return candidates[0]\n        ranks = list(range(1, len(candidates) + 1))\n        weights = [1.0 / rank for rank in ranks]\n        total = sum(weights)\n        norm = [w / total for w in weights]\n        index = self.rng.choices(range(len(candidates)), weights=norm, k=1)[0]\n        return candidates[index]\n\n    def _sample_aug(self) -> tuple[float, float, float]:\n        shift_x = self.rng.uniform(-0.08, 0.08)\n        shift_y = self.rng.uniform(-0.08, 0.08)\n        scale = 1.0 + self.rng.uniform(-0.12, 0.12)\n        return shift_x, shift_y, scale\n\n    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:\n        ref = self.refs[idx % len(self.refs)]\n        info = parse_795_reference_name(ref)\n        style = str(info.get(\"style\"))\n        shift_x, shift_y, scale = self._sample_aug()\n\n        class DummyCfg:\n            pass\n\n        local = DummyCfg()\n        local.output_size = self.cfg.slm_size\n        local.reference_output_size = self.cfg.reference_output_size\n        local.target_center_x = None\n        local.target_center_y = None\n        local.target_shift_x_m = 0.0\n        local.target_shift_y_m = 0.0\n        local.roi_diameter = self.cfg.target_sigma_px\n        local.target_sigma = self.cfg.target_sigma_px\n        local.target_type = \"target_flat_top\"\n        local.weight_type = \"gaussian_top_round\"\n        local.phase_type = \"phase_flat\"\n        local.uniformity_weight = 0.0\n        local.efficiency_weight = 0.0\n        local = apply_795_reference_metadata(local, info)\n\n        cx = (local.target_center_x if local.target_center_x is not None else self.cfg.slm_size / 2.0) + shift_x * self.cfg.slm_size\n        cy = (local.target_center_y if local.target_center_y is not None else self.cfg.slm_size / 2.0) + shift_y * self.cfg.slm_size\n        sigma = max(4.0, float(local.target_sigma) * scale)\n        roi_d = max(4.0, float(local.roi_diameter) * scale)\n        center = (cy, cx)\n\n        target_kwargs = {}\n        if getattr(local, \"target_type\", \"\") == \"target_line_flat_top\":\n            target_kwargs[\"sigma_x\"] = getattr(local, \"line_flat_top_sigma_x\", max(2.0, sigma * 0.20))\n            target_kwargs[\"sigma_y\"] = getattr(local, \"line_flat_top_sigma_y\", max(2.0, sigma * 0.20))\n        target_amp = build_target(\n            target_type=local.target_type,\n            size=self.cfg.slm_size,\n            center=center,\n            sigma=sigma,\n            charge=0,\n            device=self.cfg.device,\n            **target_kwargs,\n        )\n        target_phase = build_phase(\n            phase_type=local.phase_type,\n            size=self.cfg.slm_size,\n            center=center,\n            device=self.cfg.device,\n        )\n        weight_kwargs = {}\n        if getattr(local, \"weight_type\", \"\") == \"gaussian_top_line\":\n            weight_kwargs[\"sigma_x\"] = getattr(local, \"line_flat_top_sigma_x\", max(2.0, roi_d * 0.20))\n            weight_kwargs[\"sigma_y\"] = getattr(local, \"line_flat_top_sigma_y\", max(2.0, roi_d * 0.20))\n        roi = build_weight(\n            weight_type=local.weight_type,\n            size=self.cfg.slm_size,\n            center=center,\n            diameter=roi_d,\n            softness=self.cfg.roi_softness_px,\n            device=self.cfg.device,\n            **weight_kwargs,\n        )\n        weight = threshold_weight(roi, fraction=self.cfg.weight_threshold)\n        a_input = target_amp * weight\n        if a_input.max() > 0:\n            a_input = a_input / a_input.max()\n        phi_input = target_phase / torch.pi\n        core_mask = target_core_weight(a_input, weight, self.cfg.flat_core_threshold)\n        core_mask = (core_mask > 0).to(a_input.dtype)\n        roi_mask = (weight > 0).to(a_input.dtype)\n\n        teacher_source = self._choose_teacher_source(style)\n        hybrid_teacher = self._choose_hybrid_teacher_path(style)\n        teacher_path = hybrid_teacher if teacher_source == \"hybrid\" and hybrid_teacher is not None else ref\n        reference_phase = load_reference_phase(teacher_path, size=(self.cfg.slm_size, self.cfg.slm_size), device=self.cfg.device)\n        teacher_hologram = torch.exp(1j * reference_phase)\n        a_label, phi_label = hologram_to_position_labels(teacher_hologram, label_size=self.cfg.slm_size)\n        return {\n            \"a_input\": a_input,\n            \"phi_input\": phi_input.float(),\n            \"a_label\": a_label,\n            \"phi_label\": phi_label,\n            \"teacher_phase\": reference_phase.float(),\n            \"core_mask\": core_mask.float(),\n            \"roi_mask\": roi_mask.float(),\n            \"mode\": \"flat_top\",\n            \"reference_path\": str(teacher_path),\n            \"teacher_source\": teacher_source if teacher_source != \"hybrid\" or hybrid_teacher is not None else \"795\",\n            \"teacher_style\": style,\n        }\n\n\ndef create_dataset(cfg: Lin2025Config, num_samples: int, seed: int) -> object:\n    if cfg.target_mode == \"flat_top\":\n        return FlatTopReferenceDataset(cfg, num_samples=num_samples, seed=seed)\n    return OnTheFlyWGSDataset(cfg, num_samples=num_samples, seed=seed)\n",
  "lin2025_holography/encoding.py": "from __future__ import annotations\n\nimport math\n\nimport torch\n\n\ndef bilinear_scatter(\n    coords: torch.Tensor,\n    values: torch.Tensor,\n    size: int,\n    device: torch.device,\n) -> torch.Tensor:\n    image = torch.zeros((size, size), dtype=torch.float32, device=device)\n    x = coords[:, 0].clamp(0, size - 1 - 1e-6)\n    y = coords[:, 1].clamp(0, size - 1 - 1e-6)\n    x0 = torch.floor(x).long()\n    y0 = torch.floor(y).long()\n    x1 = torch.clamp(x0 + 1, max=size - 1)\n    y1 = torch.clamp(y0 + 1, max=size - 1)\n    wx = x - x0.float()\n    wy = y - y0.float()\n\n    image.index_put_((y0, x0), values * (1 - wx) * (1 - wy), accumulate=True)\n    image.index_put_((y0, x1), values * wx * (1 - wy), accumulate=True)\n    image.index_put_((y1, x0), values * (1 - wx) * wy, accumulate=True)\n    image.index_put_((y1, x1), values * wx * wy, accumulate=True)\n    return image\n\n\ndef encode_inputs(\n    coords: torch.Tensor,\n    phases: torch.Tensor,\n    size: int,\n    device: torch.device,\n) -> tuple[torch.Tensor, torch.Tensor]:\n    amp = bilinear_scatter(coords, torch.ones_like(phases), size=size, device=device)\n    phase_cos = bilinear_scatter(coords, torch.cos(phases), size=size, device=device)\n    phase_sin = bilinear_scatter(coords, torch.sin(phases), size=size, device=device)\n    phase = torch.atan2(phase_sin, phase_cos + 1e-9)\n    if amp.max() > 0:\n        amp = amp / amp.max()\n    phase = phase / math.pi\n    return amp, phase\n\n",
  "lin2025_holography/inference.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom ai_holography.visualization import save_field_visualizations, save_linecuts\n\nfrom .config import Lin2025Config\nfrom .dataset import create_dataset\nfrom .metrics import amplitude_l1, flat_top_metric_loss, phase_l2\nfrom .model import Lin2025HologramNet, position_to_hologram\nfrom .wgs import crop_center\n\n\ndef run_lin2025_demo(cfg: Lin2025Config, checkpoint: Path | None = None) -> dict[str, float]:\n    device = torch.device(cfg.device)\n    model = Lin2025HologramNet(hidden_channels=cfg.hidden_channels).to(device)\n    ckpt = checkpoint or (cfg.checkpoint_dir / \"lin2025_best.pt\")\n    if ckpt.exists():\n        model.load_state_dict(torch.load(ckpt, map_location=device))\n    model.eval()\n\n    sample = create_dataset(cfg, num_samples=1, seed=7)[0]\n    with torch.no_grad():\n        pred_amp, pred_phase = model(sample[\"a_input\"].unsqueeze(0), sample[\"phi_input\"].unsqueeze(0))\n    pred_amp = pred_amp.squeeze(0)\n    pred_phase = pred_phase.squeeze(0)\n\n    pred_holo = position_to_hologram(pred_amp, pred_phase)\n    label_holo = position_to_hologram(sample[\"a_label\"], sample[\"phi_label\"])\n    pred_crop = crop_center(pred_holo, cfg.slm_size)\n    label_crop = crop_center(label_holo, cfg.slm_size)\n\n    metrics = {\n        \"amp_l1\": float(amplitude_l1(pred_amp, sample[\"a_label\"]).cpu()),\n        \"phase_l2\": float(phase_l2(pred_phase, sample[\"phi_label\"]).cpu()),\n        \"hologram_mae\": float(torch.mean(torch.abs(pred_crop - label_crop)).cpu()),\n        \"checkpoint\": str(ckpt),\n    }\n    if cfg.target_mode == \"flat_top\":\n        weight = (sample[\"a_input\"] > cfg.weight_threshold).to(sample[\"a_input\"].dtype)\n        _, flat_metrics = flat_top_metric_loss(\n            pred_amp=pred_amp.unsqueeze(0),\n            pred_phase=pred_phase.unsqueeze(0),\n            target_amp=sample[\"a_input\"].unsqueeze(0),\n            target_phase=sample[\"phi_input\"].unsqueeze(0),\n            weight=weight.unsqueeze(0),\n            overlap_weight=cfg.flat_overlap_weight,\n            uniformity_weight=cfg.flat_uniformity_weight,\n            core_uniformity_weight=cfg.flat_core_uniformity_weight,\n            efficiency_weight=cfg.flat_efficiency_weight,\n            phase_weight=cfg.flat_phase_weight,\n            core_phase_weight=cfg.flat_core_phase_weight,\n            core_threshold=cfg.flat_core_threshold,\n        )\n        metrics.update(flat_metrics)\n\n    cfg.output_dir.mkdir(parents=True, exist_ok=True)\n    np.save(cfg.output_dir / \"predicted_hologram_phase.npy\", torch.angle(pred_crop).cpu().numpy())\n    (cfg.output_dir / \"metrics.json\").write_text(json.dumps(metrics, indent=2), encoding=\"utf-8\")\n    save_field_visualizations(\n        cfg.output_dir,\n        target_amp=sample[\"a_label\"],\n        target_phase=sample[\"phi_label\"] * torch.pi,\n        out_amp=pred_amp,\n        out_phase=pred_phase * torch.pi,\n        slm_phase=torch.angle(pred_crop),\n    )\n    save_linecuts(cfg.output_dir, sample[\"a_label\"], pred_amp)\n    return metrics\n",
  "lin2025_holography/metrics.py": "from __future__ import annotations\n\nimport math\n\nimport torch\nimport torch.nn.functional as F\n\nfrom ai_holography.losses import (\n    core_phase_flatness_metric,\n    core_phase_loss,\n    core_uniformity_loss,\n    efficiency_metric,\n    normalized_overlap,\n    phase_flatness_metric,\n    uniformity_loss,\n)\nfrom ai_holography.propagation import FourierSLM\nfrom ai_holography.targets import gaussian_beam\n\nfrom .model import position_to_hologram\nfrom .wgs import crop_center\n\n\ndef amplitude_l1(pred_amp: torch.Tensor, target_amp: torch.Tensor) -> torch.Tensor:\n    return torch.mean(torch.abs(pred_amp - target_amp))\n\n\ndef phase_l2(pred_phase: torch.Tensor, target_phase: torch.Tensor) -> torch.Tensor:\n    delta = torch.atan2(\n        torch.sin((pred_phase - target_phase) * math.pi),\n        torch.cos((pred_phase - target_phase) * math.pi),\n    )\n    return torch.mean(delta.square())\n\n\ndef wrapped_phase_l1(pred_phase_rad: torch.Tensor, target_phase_rad: torch.Tensor) -> torch.Tensor:\n    delta = torch.atan2(\n        torch.sin(pred_phase_rad - target_phase_rad),\n        torch.cos(pred_phase_rad - target_phase_rad),\n    )\n    return torch.mean(torch.abs(delta))\n\n\ndef weighted_wrapped_phase_l1(\n    pred_phase_rad: torch.Tensor,\n    target_phase_rad: torch.Tensor,\n    weight: torch.Tensor,\n) -> torch.Tensor:\n    delta = torch.atan2(\n        torch.sin(pred_phase_rad - target_phase_rad),\n        torch.cos(pred_phase_rad - target_phase_rad),\n    )\n    weighted = torch.abs(delta) * weight\n    denom = weight.sum().clamp_min(1e-9)\n    return weighted.sum() / denom\n\n\ndef hologram_field_error(pred_amp: torch.Tensor, pred_phase: torch.Tensor, target_amp: torch.Tensor, target_phase: torch.Tensor) -> torch.Tensor:\n    pred_holo = position_to_hologram(pred_amp, pred_phase)\n    target_holo = position_to_hologram(target_amp, target_phase)\n    pred_crop = crop_center(pred_holo, pred_amp.shape[-1])\n    target_crop = crop_center(target_holo, target_amp.shape[-1])\n    return torch.mean(torch.abs(pred_crop - target_crop))\n\n\ndef soft_efficiency_constraint(efficiency: torch.Tensor, floor: float) -> torch.Tensor:\n    margin = torch.relu(torch.tensor(floor, device=efficiency.device, dtype=efficiency.dtype) - efficiency)\n    return margin.square()\n\n\ndef flat_top_metric_loss(\n    pred_amp: torch.Tensor,\n    pred_phase: torch.Tensor,\n    target_amp: torch.Tensor,\n    target_phase: torch.Tensor,\n    weight: torch.Tensor,\n    overlap_weight: float,\n    uniformity_weight: float,\n    core_uniformity_weight: float,\n    efficiency_weight: float,\n    phase_weight: float,\n    core_phase_weight: float,\n    core_threshold: float,\n    efficiency_floor: float = 0.15,\n) -> tuple[torch.Tensor, dict[str, float]]:\n    target_phase_rad = target_phase * math.pi\n    pred_phase_rad = pred_phase * math.pi\n    overlap = normalized_overlap(target_amp, target_phase_rad, pred_amp, pred_phase_rad, weight)\n    overlap_loss = (1.0 - overlap).square()\n    uniformity = uniformity_loss(target_amp, pred_amp, weight)\n    core_uniformity = core_uniformity_loss(target_amp, pred_amp, weight, core_threshold)\n    efficiency = efficiency_metric(weight, pred_amp)\n    efficiency_constraint = soft_efficiency_constraint(efficiency, efficiency_floor)\n    phase_loss = core_phase_loss(target_phase_rad, pred_phase_rad, target_amp, weight, core_threshold)\n    phase_flatness = phase_flatness_metric(target_phase_rad, pred_phase_rad, weight)\n    core_phase_flatness = core_phase_flatness_metric(target_phase_rad, pred_phase_rad, target_amp, weight, core_threshold)\n    total = (\n        overlap_weight * overlap_loss\n        + uniformity_weight * uniformity\n        + core_uniformity_weight * core_uniformity\n        + efficiency_weight * efficiency_constraint\n        + phase_weight * phase_flatness\n        + core_phase_weight * phase_loss\n    )\n    metrics = {\n        \"overlap\": float(overlap.detach().cpu()),\n        \"uniformity_loss\": float(uniformity.detach().cpu()),\n        \"core_uniformity_loss\": float(core_uniformity.detach().cpu()),\n        \"efficiency\": float(efficiency.detach().cpu()),\n        \"efficiency_constraint\": float(efficiency_constraint.detach().cpu()),\n        \"phase_flatness\": float(phase_flatness.detach().cpu()),\n        \"core_phase_flatness\": float(core_phase_flatness.detach().cpu()),\n    }\n    return total, metrics\n\n\ndef hybrid_init_score(\n    pred_hologram_phase: torch.Tensor,\n    target_amp_small: torch.Tensor,\n    target_phase_small: torch.Tensor,\n    beam_sigma_px: float,\n    overlap_weight: float,\n    uniformity_weight: float,\n    core_uniformity_weight: float,\n    efficiency_weight: float,\n    core_phase_weight: float,\n    core_threshold: float,\n    efficiency_floor: float = 0.15,\n) -> tuple[torch.Tensor, dict[str, float]]:\n    batch = pred_hologram_phase.shape[0]\n    slm_size = pred_hologram_phase.shape[-1]\n    out_size = 2 * slm_size\n    device = pred_hologram_phase.device\n    propagator = FourierSLM(slm_size, out_size).to(device)\n    beam_2d = gaussian_beam(slm_size, beam_sigma_px, beam_sigma_px, str(device))\n    beam = beam_2d.unsqueeze(0).expand(batch, -1, -1)\n    target_amp = F.interpolate(target_amp_small.unsqueeze(1), size=(out_size, out_size), mode=\"bilinear\", align_corners=False).squeeze(1)\n    target_phase = F.interpolate(target_phase_small.unsqueeze(1), size=(out_size, out_size), mode=\"bilinear\", align_corners=False).squeeze(1) * math.pi\n    weight = (target_amp > 0.05 * target_amp.amax(dim=(-2, -1), keepdim=True)).to(target_amp.dtype)\n    out_amp, out_phase = propagator(beam, pred_hologram_phase)\n    score, metrics = flat_top_metric_loss(\n        pred_amp=out_amp,\n        pred_phase=out_phase / math.pi,\n        target_amp=target_amp,\n        target_phase=target_phase / math.pi,\n        weight=weight,\n        overlap_weight=overlap_weight,\n        uniformity_weight=uniformity_weight,\n        core_uniformity_weight=core_uniformity_weight,\n        efficiency_weight=efficiency_weight,\n        phase_weight=0.0,\n        core_phase_weight=core_phase_weight,\n        core_threshold=core_threshold,\n        efficiency_floor=efficiency_floor,\n    )\n    return score, metrics\n",
  "lin2025_holography/model.py": "from __future__ import annotations\n\nimport math\n\nimport torch\nfrom torch import nn\n\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, channels: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(channels, channels, kernel_size=3, padding=1),\n            nn.BatchNorm2d(channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(channels, channels, kernel_size=3, padding=1),\n            nn.BatchNorm2d(channels),\n        )\n        self.act = nn.ReLU(inplace=True)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.act(x + self.net(x))\n\n\nclass Lin2025HologramNet(nn.Module):\n    \"\"\"\n    Position-domain predictor inspired by Lin et al. 2025.\n    Input: amplitude/phase images in the position domain.\n    Output: amplitude/phase labels in the position domain.\n    \"\"\"\n\n    def __init__(self, input_channels: int = 4, hidden_channels: int = 48, num_blocks: int = 4):\n        super().__init__()\n        self.stem = nn.Sequential(\n            nn.Conv2d(input_channels, hidden_channels, kernel_size=3, padding=1),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1),\n            nn.ReLU(inplace=True),\n        )\n        self.blocks = nn.Sequential(*[ResidualBlock(hidden_channels) for _ in range(num_blocks)])\n        self.amp_head = nn.Sequential(\n            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(hidden_channels, 1, kernel_size=1),\n        )\n        self.phase_head = nn.Sequential(\n            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(hidden_channels, 1, kernel_size=1),\n        )\n\n    def forward(\n        self,\n        a_input: torch.Tensor,\n        phi_input: torch.Tensor,\n        core_mask: torch.Tensor | None = None,\n        roi_mask: torch.Tensor | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        if core_mask is None:\n            core_mask = torch.zeros_like(a_input)\n        if roi_mask is None:\n            roi_mask = (a_input > 0).to(a_input.dtype)\n        x = torch.stack([a_input, phi_input, core_mask, roi_mask], dim=1)\n        features = self.blocks(self.stem(x))\n        amp = torch.sigmoid(self.amp_head(features).squeeze(1))\n        phase = torch.tanh(self.phase_head(features).squeeze(1))\n        return amp, phase\n\n\ndef position_to_hologram(pred_amp: torch.Tensor, pred_phase: torch.Tensor) -> torch.Tensor:\n    field = pred_amp * torch.exp(1j * (pred_phase * math.pi))\n    return torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field)))\n",
  "lin2025_holography/predictor.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport torch\nimport torch.nn.functional as F\n\nfrom .config import Lin2025Config\nfrom .model import Lin2025HologramNet, position_to_hologram\nfrom .wgs import crop_center\nfrom ai_holography.losses import target_core_weight\n\n\ndef _resolve_model_shape(cfg: Lin2025Config, checkpoint: Path | None) -> tuple[int, int, int]:\n    input_channels = cfg.input_channels\n    hidden_channels = cfg.hidden_channels\n    num_blocks = cfg.num_residual_blocks\n    candidate_dirs: list[Path] = []\n    if checkpoint is not None:\n        candidate_dirs.append(checkpoint.parent)\n    candidate_dirs.append(cfg.checkpoint_dir)\n    for directory in candidate_dirs:\n        history_path = Path(directory) / \"lin2025_training_history.json\"\n        if not history_path.exists():\n            continue\n        try:\n            history = json.loads(history_path.read_text(encoding=\"utf-8\"))\n            config_data = history.get(\"config\", {})\n            input_channels = int(config_data.get(\"input_channels\", input_channels))\n            hidden_channels = int(config_data.get(\"hidden_channels\", hidden_channels))\n            num_blocks = int(config_data.get(\"num_residual_blocks\", num_blocks))\n            return input_channels, hidden_channels, num_blocks\n        except Exception:\n            continue\n    return input_channels, hidden_channels, num_blocks\n\n\ndef load_lin2025_model(cfg: Lin2025Config, checkpoint: Path | None = None) -> tuple[Lin2025HologramNet, Path | None]:\n    device = torch.device(cfg.device)\n    resolved_checkpoint = checkpoint\n    if resolved_checkpoint is None:\n        resolved_checkpoint = cfg.checkpoint_dir / \"lin2025_best_hybrid_polish.pt\"\n    input_channels, hidden_channels, num_blocks = _resolve_model_shape(cfg, resolved_checkpoint)\n    model = Lin2025HologramNet(\n        input_channels=input_channels,\n        hidden_channels=hidden_channels,\n        num_blocks=num_blocks,\n    ).to(device)\n    ckpt = resolved_checkpoint\n    if not ckpt.exists():\n        fallback = cfg.checkpoint_dir / \"lin2025_best_quality.pt\"\n        if fallback.exists():\n            ckpt = fallback\n        else:\n            legacy = cfg.checkpoint_dir / \"lin2025_best_init_for_hybrid.pt\"\n            ckpt = legacy if legacy.exists() else (cfg.checkpoint_dir / \"lin2025_best.pt\")\n    loaded = ckpt if ckpt.exists() else None\n    if loaded is not None:\n        model.load_state_dict(torch.load(loaded, map_location=device))\n    model.eval()\n    return model, loaded\n\n\ndef predict_hologram_phase(\n    cfg: Lin2025Config,\n    target_amp: torch.Tensor,\n    target_phase: torch.Tensor,\n    checkpoint: Path | None = None,\n) -> tuple[torch.Tensor, Path | None]:\n    device = torch.device(cfg.device)\n    model, loaded = load_lin2025_model(cfg, checkpoint=checkpoint)\n    size = cfg.slm_size\n    if target_amp.ndim == 2:\n        target_amp = target_amp.unsqueeze(0)\n    if target_phase.ndim == 2:\n        target_phase = target_phase.unsqueeze(0)\n    amp_small = F.interpolate(target_amp.unsqueeze(1), size=(size, size), mode=\"bilinear\", align_corners=False).squeeze(1).to(device)\n    phase_small = F.interpolate(target_phase.unsqueeze(1), size=(size, size), mode=\"bilinear\", align_corners=False).squeeze(1).to(device)\n    roi_mask = (amp_small > 0.05 * amp_small.amax(dim=(-2, -1), keepdim=True)).to(amp_small.dtype)\n    core_mask = target_core_weight(amp_small, roi_mask, cfg.flat_core_threshold)\n    with torch.no_grad():\n        pred_amp, pred_phase = model(\n            amp_small,\n            phase_small / torch.pi,\n            core_mask=core_mask,\n            roi_mask=roi_mask,\n        )\n    hologram = position_to_hologram(pred_amp, pred_phase)\n    crop = crop_center(hologram.squeeze(0), size)\n    return torch.angle(crop), loaded\n",
  "lin2025_holography/training.py": "from __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport torch\n\nfrom .config import Lin2025Config\nfrom .dataset import create_dataset\nfrom .metrics import (\n    amplitude_l1,\n    flat_top_metric_loss,\n    hologram_field_error,\n    hybrid_init_score,\n    phase_l2,\n    soft_efficiency_constraint,\n    weighted_wrapped_phase_l1,\n)\nfrom .model import Lin2025HologramNet, position_to_hologram\n\n\ndef _batchify(samples: list[dict[str, torch.Tensor]]) -> dict[str, torch.Tensor]:\n    keys = (\"a_input\", \"phi_input\", \"a_label\", \"phi_label\")\n    batch = {key: torch.stack([sample[key] for sample in samples], dim=0) for key in keys}\n    if \"teacher_phase\" in samples[0]:\n        batch[\"teacher_phase\"] = torch.stack([sample[\"teacher_phase\"] for sample in samples], dim=0)\n    if \"core_mask\" in samples[0]:\n        batch[\"core_mask\"] = torch.stack([sample[\"core_mask\"] for sample in samples], dim=0)\n    if \"roi_mask\" in samples[0]:\n        batch[\"roi_mask\"] = torch.stack([sample[\"roi_mask\"] for sample in samples], dim=0)\n    if \"mode\" in samples[0]:\n        batch[\"mode\"] = samples[0][\"mode\"]\n    return batch\n\n\ndef _teacher_phase_distillation(\n    cfg: Lin2025Config,\n    pred_holo_phase: torch.Tensor,\n    batch: dict[str, torch.Tensor],\n) -> tuple[torch.Tensor, dict[str, float]]:\n    teacher_phase = batch[\"teacher_phase\"].to(pred_holo_phase.device)\n    core_weight = batch.get(\"core_mask\")\n    roi_weight = batch.get(\"roi_mask\")\n    if core_weight is not None:\n        core_weight = core_weight.to(pred_holo_phase.device)\n    if roi_weight is not None:\n        roi_weight = roi_weight.to(pred_holo_phase.device)\n\n    teacher_loss = torch.tensor(0.0, device=pred_holo_phase.device)\n    metrics = {\n        \"teacher_phase_loss\": 0.0,\n        \"core_teacher_phase_loss\": 0.0,\n    }\n    if roi_weight is not None:\n        teacher_phase_loss = weighted_wrapped_phase_l1(pred_holo_phase, teacher_phase, roi_weight)\n        teacher_loss = teacher_loss + cfg.hologram_roi_phase_imitation_weight * teacher_phase_loss\n        metrics[\"teacher_phase_loss\"] = float(teacher_phase_loss.detach().cpu())\n    if core_weight is not None:\n        core_teacher_phase_loss = weighted_wrapped_phase_l1(pred_holo_phase, teacher_phase, core_weight)\n        teacher_loss = teacher_loss + cfg.hologram_core_phase_imitation_weight * core_teacher_phase_loss\n        metrics[\"core_teacher_phase_loss\"] = float(core_teacher_phase_loss.detach().cpu())\n    return teacher_loss, metrics\n\n\ndef train_lin2025_model(cfg: Lin2025Config) -> Path:\n    device = torch.device(cfg.device)\n    cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    model = Lin2025HologramNet(\n        input_channels=cfg.input_channels,\n        hidden_channels=cfg.hidden_channels,\n        num_blocks=cfg.num_residual_blocks,\n    ).to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\n        optimizer,\n        mode=\"min\",\n        factor=cfg.scheduler_factor,\n        patience=cfg.scheduler_patience,\n        min_lr=cfg.scheduler_min_lr,\n    )\n\n    train_set = create_dataset(cfg, num_samples=cfg.train_samples, seed=42)\n    val_set = create_dataset(cfg, num_samples=cfg.val_samples, seed=4242)\n\n    best_val = float(\"inf\")\n    best_quality_score = float(\"inf\")\n    best_hybrid_polish_score = float(\"inf\")\n    best_path = cfg.checkpoint_dir / \"lin2025_best_hybrid_polish.pt\"\n    best_supervised_path = cfg.checkpoint_dir / \"lin2025_best_supervised.pt\"\n    best_quality_path = cfg.checkpoint_dir / \"lin2025_best_quality.pt\"\n    best_hybrid_polish_path = cfg.checkpoint_dir / \"lin2025_best_hybrid_polish.pt\"\n    latest_path = cfg.checkpoint_dir / \"lin2025_latest.pt\"\n    history: list[dict[str, float]] = []\n\n    for epoch in range(1, cfg.epochs + 1):\n        model.train()\n        train_loss_sum = 0.0\n        start = time.perf_counter()\n        train_teacher_phase_sum = 0.0\n        train_core_teacher_phase_sum = 0.0\n        for start_idx in range(0, len(train_set), cfg.batch_size):\n            batch_samples = [train_set[idx] for idx in range(start_idx, min(start_idx + cfg.batch_size, len(train_set)))]\n            batch = _batchify(batch_samples)\n            pred_amp, pred_phase = model(\n                batch[\"a_input\"],\n                batch[\"phi_input\"],\n                core_mask=batch.get(\"core_mask\"),\n                roi_mask=batch.get(\"roi_mask\"),\n            )\n            loss_amp = amplitude_l1(pred_amp, batch[\"a_label\"])\n            loss_phase = phase_l2(pred_phase, batch[\"phi_label\"])\n            loss_holo = hologram_field_error(pred_amp, pred_phase, batch[\"a_label\"], batch[\"phi_label\"])\n            loss = loss_amp + loss_phase + 0.1 * loss_holo\n            if \"teacher_phase\" in batch:\n                pred_holo_phase = torch.angle(position_to_hologram(pred_amp, pred_phase))\n                teacher_loss, teacher_metrics = _teacher_phase_distillation(cfg, pred_holo_phase, batch)\n                loss = loss + teacher_loss\n                train_teacher_phase_sum += teacher_metrics[\"teacher_phase_loss\"]\n                train_core_teacher_phase_sum += teacher_metrics[\"core_teacher_phase_loss\"]\n            if cfg.target_mode == \"flat_top\":\n                weight = (batch[\"a_input\"] > cfg.weight_threshold).to(batch[\"a_input\"].dtype)\n                metric_loss, _ = flat_top_metric_loss(\n                    pred_amp=pred_amp,\n                    pred_phase=pred_phase,\n                    target_amp=batch[\"a_input\"],\n                    target_phase=batch[\"phi_input\"],\n                    weight=weight,\n                    overlap_weight=cfg.flat_overlap_weight,\n                    uniformity_weight=cfg.flat_uniformity_weight,\n                    core_uniformity_weight=cfg.flat_core_uniformity_weight,\n                    efficiency_weight=cfg.flat_efficiency_weight,\n                    phase_weight=cfg.flat_phase_weight,\n                    core_phase_weight=cfg.flat_core_phase_weight,\n                    core_threshold=cfg.flat_core_threshold,\n                    efficiency_floor=cfg.efficiency_floor,\n                )\n                loss = loss + metric_loss\n            if cfg.skip_nonfinite_batches and not torch.isfinite(loss):\n                optimizer.zero_grad(set_to_none=True)\n                continue\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            if cfg.grad_clip_norm > 0:\n                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)\n            optimizer.step()\n            train_loss_sum += float(loss.detach().cpu())\n\n        model.eval()\n        val_loss_sum = 0.0\n        val_metric_sums = {\n            \"uniformity_loss\": 0.0,\n            \"core_uniformity_loss\": 0.0,\n            \"efficiency\": 0.0,\n            \"phase_flatness\": 0.0,\n            \"core_phase_flatness\": 0.0,\n            \"overlap\": 0.0,\n            \"teacher_phase_loss\": 0.0,\n            \"core_teacher_phase_loss\": 0.0,\n        }\n        hybrid_polish_score_sum = 0.0\n        with torch.no_grad():\n            for start_idx in range(0, len(val_set), cfg.batch_size):\n                batch_samples = [val_set[idx] for idx in range(start_idx, min(start_idx + cfg.batch_size, len(val_set)))]\n                batch = _batchify(batch_samples)\n                pred_amp, pred_phase = model(\n                    batch[\"a_input\"],\n                    batch[\"phi_input\"],\n                    core_mask=batch.get(\"core_mask\"),\n                    roi_mask=batch.get(\"roi_mask\"),\n                )\n                loss_amp = amplitude_l1(pred_amp, batch[\"a_label\"])\n                loss_phase = phase_l2(pred_phase, batch[\"phi_label\"])\n                loss_holo = hologram_field_error(pred_amp, pred_phase, batch[\"a_label\"], batch[\"phi_label\"])\n                loss = loss_amp + loss_phase + 0.1 * loss_holo\n                if \"teacher_phase\" in batch:\n                    pred_holo_phase = torch.angle(position_to_hologram(pred_amp, pred_phase))\n                    teacher_loss, teacher_metrics = _teacher_phase_distillation(cfg, pred_holo_phase, batch)\n                    loss = loss + teacher_loss\n                    val_metric_sums[\"teacher_phase_loss\"] += teacher_metrics[\"teacher_phase_loss\"]\n                    val_metric_sums[\"core_teacher_phase_loss\"] += teacher_metrics[\"core_teacher_phase_loss\"]\n                metric_values = None\n                if cfg.target_mode == \"flat_top\":\n                    weight = (batch[\"a_input\"] > cfg.weight_threshold).to(batch[\"a_input\"].dtype)\n                    metric_loss, metric_values = flat_top_metric_loss(\n                        pred_amp=pred_amp,\n                        pred_phase=pred_phase,\n                        target_amp=batch[\"a_input\"],\n                        target_phase=batch[\"phi_input\"],\n                        weight=weight,\n                        overlap_weight=cfg.flat_overlap_weight,\n                        uniformity_weight=cfg.flat_uniformity_weight,\n                        core_uniformity_weight=cfg.flat_core_uniformity_weight,\n                        efficiency_weight=cfg.flat_efficiency_weight,\n                        phase_weight=cfg.flat_phase_weight,\n                        core_phase_weight=cfg.flat_core_phase_weight,\n                        core_threshold=cfg.flat_core_threshold,\n                        efficiency_floor=cfg.efficiency_floor,\n                    )\n                    loss = loss + metric_loss\n                val_loss_sum += float(loss.detach().cpu())\n                if metric_values is not None:\n                    for key in (\n                        \"uniformity_loss\",\n                        \"core_uniformity_loss\",\n                        \"efficiency\",\n                        \"phase_flatness\",\n                        \"core_phase_flatness\",\n                        \"overlap\",\n                    ):\n                        val_metric_sums[key] += float(metric_values[key])\n                    pred_holo = torch.angle(position_to_hologram(pred_amp, pred_phase))\n                    init_score, _ = hybrid_init_score(\n                        pred_hologram_phase=pred_holo,\n                        target_amp_small=batch[\"a_input\"],\n                        target_phase_small=batch[\"phi_input\"],\n                        beam_sigma_px=cfg.beam_sigma_px,\n                        overlap_weight=cfg.hybrid_init_overlap_weight,\n                        uniformity_weight=cfg.hybrid_init_uniformity_weight,\n                        core_uniformity_weight=cfg.hybrid_init_core_uniformity_weight,\n                        efficiency_weight=cfg.hybrid_init_efficiency_weight,\n                        core_phase_weight=cfg.hybrid_init_core_phase_weight,\n                        core_threshold=cfg.flat_core_threshold,\n                    )\n                    hybrid_polish_score_sum += float(init_score)\n\n        train_loss = train_loss_sum / max(1, len(range(0, len(train_set), cfg.batch_size)))\n        val_loss = val_loss_sum / max(1, len(range(0, len(val_set), cfg.batch_size)))\n        avg_metrics = {\n            key: value / max(1, len(range(0, len(val_set), cfg.batch_size)))\n            for key, value in val_metric_sums.items()\n        }\n        avg_hybrid_polish_score = hybrid_polish_score_sum / max(1, len(range(0, len(val_set), cfg.batch_size)))\n        quality_score = (\n            cfg.metric_score_core_uniformity_weight * avg_metrics[\"core_uniformity_loss\"]\n            + cfg.metric_score_core_phase_weight * avg_metrics[\"core_phase_flatness\"]\n            + cfg.metric_score_efficiency_weight * float(soft_efficiency_constraint(torch.tensor(avg_metrics[\"efficiency\"]), cfg.efficiency_floor))\n            + cfg.metric_score_uniformity_weight * avg_metrics[\"uniformity_loss\"]\n        ) if cfg.target_mode == \"flat_top\" else val_loss\n        epoch_time = time.perf_counter() - start\n        history.append({\"epoch\": float(epoch), \"train_loss\": train_loss, \"supervised_loss\": val_loss, \"quality_score\": quality_score, \"hybrid_polish_score\": avg_hybrid_polish_score, **avg_metrics, \"epoch_sec\": epoch_time})\n        history[-1][\"train_teacher_phase_loss\"] = train_teacher_phase_sum / max(1, len(range(0, len(train_set), cfg.batch_size)))\n        history[-1][\"train_core_teacher_phase_loss\"] = train_core_teacher_phase_sum / max(1, len(range(0, len(train_set), cfg.batch_size)))\n        history[-1][\"learning_rate\"] = float(optimizer.param_groups[0][\"lr\"])\n        print(\n            f\"epoch {epoch:02d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | \"\n            f\"quality={quality_score:.6f} | polish={avg_hybrid_polish_score:.6f} | \"\n            f\"core_teacher={avg_metrics['core_teacher_phase_loss']:.6f} | \"\n            f\"lr={optimizer.param_groups[0]['lr']:.2e} | time={epoch_time:.2f}s\"\n        )\n        scheduler.step(quality_score)\n        torch.save(model.state_dict(), latest_path)\n        if val_loss < best_val:\n            best_val = val_loss\n            torch.save(model.state_dict(), best_supervised_path)\n        if quality_score < best_quality_score:\n            best_quality_score = quality_score\n            torch.save(model.state_dict(), best_quality_path)\n        if avg_hybrid_polish_score < best_hybrid_polish_score:\n            best_hybrid_polish_score = avg_hybrid_polish_score\n            torch.save(model.state_dict(), best_hybrid_polish_path)\n            torch.save(model.state_dict(), best_path)\n\n    metadata = {\n        \"config\": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()},\n        \"best_val_loss\": best_val,\n        \"best_quality_score\": best_quality_score,\n        \"best_hybrid_polish_score\": best_hybrid_polish_score,\n        \"best_checkpoint\": str(best_path),\n        \"best_supervised_checkpoint\": str(best_supervised_path),\n        \"best_quality_checkpoint\": str(best_quality_path),\n        \"best_hybrid_polish_checkpoint\": str(best_hybrid_polish_path),\n        \"history\": history,\n    }\n    (cfg.checkpoint_dir / \"lin2025_training_history.json\").write_text(json.dumps(metadata, indent=2), encoding=\"utf-8\")\n    if best_hybrid_polish_path.exists():\n        legacy_best = cfg.checkpoint_dir / \"lin2025_best.pt\"\n        legacy_init = cfg.checkpoint_dir / \"lin2025_best_init_for_hybrid.pt\"\n        torch.save(torch.load(best_hybrid_polish_path, map_location=\"cpu\"), legacy_best)\n        torch.save(torch.load(best_hybrid_polish_path, map_location=\"cpu\"), legacy_init)\n    return best_path\n",
  "lin2025_holography/wgs.py": "from __future__ import annotations\n\nimport math\n\nimport torch\n\n\ndef gaussian_beam(size: int, sigma: float, device: torch.device) -> torch.Tensor:\n    axis = torch.arange(size, dtype=torch.float32, device=device)\n    y, x = torch.meshgrid(axis, axis, indexing=\"ij\")\n    c = size / 2.0\n    beam = torch.exp(-2.0 * (((x - c) / sigma) ** 2 + ((y - c) / sigma) ** 2))\n    return beam / beam.max().clamp_min(1e-9)\n\n\ndef make_target_field(\n    coords: torch.Tensor,\n    phases: torch.Tensor,\n    size: int,\n    sigma: float,\n    device: torch.device,\n) -> torch.Tensor:\n    axis = torch.arange(size, dtype=torch.float32, device=device)\n    y, x = torch.meshgrid(axis, axis, indexing=\"ij\")\n    field = torch.zeros((size, size), dtype=torch.complex64, device=device)\n    for idx in range(coords.shape[0]):\n        cx, cy = coords[idx]\n        amp = torch.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2.0 * sigma**2))\n        field = field + amp * torch.exp(1j * phases[idx])\n    return field\n\n\ndef weighted_gs_hologram(\n    coords: torch.Tensor,\n    phases: torch.Tensor,\n    slm_size: int,\n    oversampled_size: int,\n    trap_sigma: float,\n    beam_sigma: float,\n    iterations: int,\n    device: torch.device,\n) -> torch.Tensor:\n    scale = oversampled_size / slm_size\n    coords_os = coords * scale\n    target = make_target_field(coords_os, phases, oversampled_size, trap_sigma * scale, device)\n    target_amp = torch.abs(target)\n    target_phase = torch.angle(target)\n    beam = gaussian_beam(oversampled_size, beam_sigma * scale, device)\n\n    slm_phase = 2.0 * math.pi * torch.rand((oversampled_size, oversampled_size), device=device)\n    weights = torch.ones((coords.shape[0],), dtype=torch.float32, device=device)\n\n    for _ in range(iterations):\n        slm_field = beam * torch.exp(1j * slm_phase)\n        out_field = torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(slm_field)))\n        out_amp = torch.abs(out_field)\n        out_phase = torch.angle(out_field)\n\n        # Update per-trap weights from sampled trap amplitudes.\n        sample_x = coords_os[:, 0].round().long().clamp(0, oversampled_size - 1)\n        sample_y = coords_os[:, 1].round().long().clamp(0, oversampled_size - 1)\n        measured = out_amp[sample_y, sample_x].clamp_min(1e-6)\n        weights = weights * (measured.mean() / measured)\n\n        weighted_target_amp = torch.zeros_like(target_amp)\n        weighted_target_amp[sample_y, sample_x] = weights\n        weighted_target_amp = torch.maximum(weighted_target_amp, target_amp * 0.05)\n\n        constrained = weighted_target_amp * torch.exp(1j * target_phase)\n        back = torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(constrained)))\n        slm_phase = torch.angle(back)\n\n    final_field = beam * torch.exp(1j * slm_phase)\n    return final_field\n\n\ndef crop_center(field: torch.Tensor, size: int) -> torch.Tensor:\n    current = field.shape[-1]\n    start = (current - size) // 2\n    end = start + size\n    return field[start:end, start:end]\n\n\ndef hologram_to_position_labels(hologram: torch.Tensor, label_size: int) -> tuple[torch.Tensor, torch.Tensor]:\n    cropped = crop_center(hologram, label_size)\n    pos_field = torch.fft.ifftshift(torch.fft.ifft2(torch.fft.fftshift(cropped)))\n    amp = torch.abs(pos_field)\n    phase = torch.angle(pos_field)\n    amp = amp / amp.max().clamp_min(1e-9)\n    phase = phase / math.pi\n    return amp.float(), phase.float()\n\n"
}

for rel_path, text in project_files.items():
    full_path = PROJECT_ROOT / rel_path
    full_path.parent.mkdir(parents=True, exist_ok=True)
    full_path.write_text(text, encoding="utf-8")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("written files =", len(project_files))


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("written files =", len(project_files))

for rel_path, text in project_files.items():
    full_path = PROJECT_ROOT / rel_path
    full_path.parent.mkdir(parents=True, exist_ok=True)
    full_path.write_text(text, encoding="utf-8")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("written files =", len(project_files))


In [ ]:
import torch, json
print('torch=', torch.__version__)
print('cuda=', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu=', torch.cuda.get_device_name(0))


## 生成 795 风格参考相位
这一步会在运行时自动生成 `ftl_gen/795_*.npy`，不需要额外上传参考文件。


In [ ]:
from pathlib import Path
import numpy as np
import torch

from ai_holography.camera_loop import save_measured_intensity
from ai_holography.config import HolographyConfig
from ai_holography.references import parse_795_reference_name, apply_795_reference_metadata
from ai_holography.pipeline import AIHolographyPipeline
from ai_holography.experimental_heuristics import physical_phase_guess
from ai_holography.targets import quadratic_phase_guess
from ai_holography.losses import composite_loss
from ai_holography.propagation import wrap_phase

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REF_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REF_DIR = Path("ftl_gen")
REF_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_NAMES = [
    "795_gaussian_d=160_dx=67_dy=-173.npy",
    "795_round_top_d=160_dx=67_dy=-173.npy",
    "795_flat_top_d=160_sx=30_sy=30_dx=67_dy=-173.npy",
]


def generate_reference_phase(filename: str, steps: int = 140, lbfgs_steps: int = 20) -> Path:
    out_path = REF_DIR / filename
    if out_path.exists():
        print("exists:", out_path)
        return out_path

    info = parse_795_reference_name(filename)
    cfg = HolographyConfig(run_name=f"gen_{info['style']}")
    cfg.slm_size = 256
    cfg.output_size = 1024
    cfg.beam_sigma_x = 40.0
    cfg.beam_sigma_y = 40.0
    cfg.device = REF_DEVICE
    cfg = apply_795_reference_metadata(cfg, info)
    cfg.phase_type = "phase_flat"
    cfg.reference_phase_path = None
    cfg.auto_load_best_checkpoint = False
    cfg.checkpoint = None

    pipe = AIHolographyPipeline(cfg)
    problem = pipe.build_problem()

    init_phase = physical_phase_guess(
        slm_shape=(cfg.slm_size, cfg.slm_size),
        dx_m=cfg.target_shift_x_m,
        dy_m=cfg.target_shift_y_m,
        wavelength_m=cfg.wavelength_m,
        focal_length_m=cfg.focal_length_m,
        slm_pixel_pitch_m=cfg.slm_pixel_pitch_m,
        magnification=cfg.magnification,
        aspect=cfg.init_aspect,
        curvature=cfg.init_curvature,
        cone=cfg.init_cone,
        device=cfg.device,
    ) + quadratic_phase_guess(
        cfg.slm_size,
        tilt=cfg.init_tilt,
        aspect=cfg.init_aspect,
        curvature=cfg.init_curvature,
        angle=cfg.init_angle,
        cone=cfg.init_cone,
        device=cfg.device,
    )

    phase = torch.nn.Parameter(init_phase.clone())
    optimizer = torch.optim.Adam([phase], lr=0.03)

    if info["style"] == "gaussian":
        core_uniformity_weight = 0.10
        core_phase_weight = 0.02
        uniformity_weight = 0.05
        efficiency_weight = 0.20
    elif info["style"] == "round_top":
        core_uniformity_weight = 0.25
        core_phase_weight = 0.05
        uniformity_weight = 0.18
        efficiency_weight = 0.18
    else:
        core_uniformity_weight = 0.45
        core_phase_weight = 0.10
        uniformity_weight = 0.25
        efficiency_weight = 0.15

    best_loss = float("inf")
    best_phase = init_phase.detach().clone()
    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)
        wrapped = wrap_phase(phase)
        out_amp, out_phase = pipe.propagator(problem["beam"], wrapped)
        loss, _ = composite_loss(
            target_amp=problem["target_amp"],
            target_phase=problem["target_phase"],
            out_amp=out_amp,
            out_phase=out_phase,
            weight=problem["weight"],
            slm_phase=wrapped,
            overlap_weight=1.0,
            intensity_weight=0.1,
            phase_weight=0.05,
            uniformity_weight=uniformity_weight,
            efficiency_weight=efficiency_weight,
            core_uniformity_weight=core_uniformity_weight,
            core_phase_weight=core_phase_weight,
            core_region_threshold=0.78,
            smoothness_weight=1e-4,
            efficiency_floor=cfg.efficiency_floor,
        )
        loss.backward()
        optimizer.step()
        loss_value = float(loss.detach().cpu())
        if loss_value < best_loss:
            best_loss = loss_value
            best_phase = wrap_phase(phase.detach()).clone()
        if step % 20 == 0:
            print(f"  [{info['style']}] Adam step {step}/{steps} loss={loss_value:.6f}")

    phase = torch.nn.Parameter(best_phase.clone())
    lbfgs = torch.optim.LBFGS([phase], max_iter=lbfgs_steps, line_search_fn="strong_wolfe")

    def closure():
        lbfgs.zero_grad(set_to_none=True)
        wrapped = wrap_phase(phase)
        out_amp, out_phase = pipe.propagator(problem["beam"], wrapped)
        loss, _ = composite_loss(
            target_amp=problem["target_amp"],
            target_phase=problem["target_phase"],
            out_amp=out_amp,
            out_phase=out_phase,
            weight=problem["weight"],
            slm_phase=wrapped,
            overlap_weight=1.0,
            intensity_weight=0.1,
            phase_weight=0.05,
            uniformity_weight=uniformity_weight,
            efficiency_weight=efficiency_weight,
            core_uniformity_weight=core_uniformity_weight,
            core_phase_weight=core_phase_weight,
            core_region_threshold=0.78,
            smoothness_weight=1e-4,
            efficiency_floor=cfg.efficiency_floor,
        )
        loss.backward()
        return loss

    lbfgs.step(closure)
    final_phase = wrap_phase(phase.detach()).cpu().numpy().astype(np.float32)
    np.save(out_path, final_phase)
    print("generated:", out_path, "on", REF_DEVICE)
    return out_path


for ref_name in REFERENCE_NAMES:
    generate_reference_phase(ref_name)

print("reference files:")
for p in sorted(REF_DIR.glob("795*.npy")):
    print("-", p.name)


## 跳过旧版 `PhaseInitNet`
为了加快 AutoDL 训练，这个版本默认不再训练旧版 `PhaseInitNet`。

后续 `795_only` 基线会按下面这套方式构造：
- `round_top`: `reference`
- `flat_top`: `reference + warm_start`

这样可以把 `795` 作为稳定基线保留下来，同时把主要注意力放在后续优化路线。


In [ ]:
print('Skipping legacy PhaseInitNet training in the fast AutoDL notebook.')


In [ ]:
# Fix: _build_init_phase should not force neural_init when only reference is requested
from pathlib import Path
import torch
import torch.nn.functional as F
from ai_holography.runner import HybridHolographyRunner
from ai_holography.references import load_reference_phase
from lin2025_holography.predictor import predict_hologram_phase

_original_build_init_phase = HybridHolographyRunner._build_init_phase

def _fixed_build_init_phase(self, problem, use_warm_start=False):
    sources = set(self.cfg.init_candidate_sources)
    # Get device from problem tensors if available
    device = self.cfg.device
    if 'beam' in problem:
        device = problem['beam'].device
    neural_init = None
    if 'neural' in sources:
        neural_init = self.pipeline.predict_initial_phase(
            problem['target_amp'], problem['target_phase'], problem['weight']
        )
        if neural_init.device != device:
            neural_init = neural_init.to(device)
    candidate_phases = {}
    best_phase = None
    best_overlap = float('-inf')
    best_score = float('inf')
    best_mix = 0.0
    best_source = 'neural'
    if 'neural' in sources and neural_init is not None:
        candidate_phases['neural'] = neural_init
        best_phase = neural_init
        best_overlap = self._estimate_overlap(problem, neural_init)
        best_score = self._estimate_init_score(problem, neural_init)
    if 'reference' in sources and self.cfg.reference_phase_path is not None and Path(self.cfg.reference_phase_path).exists():
        ref_phase = load_reference_phase(
            self.cfg.reference_phase_path,
            size=(self.cfg.slm_size, self.cfg.slm_size),
            device=device,
        )
        candidate_phases['reference'] = ref_phase
        ref_overlap = self._estimate_overlap(problem, ref_phase)
        ref_score = self._estimate_init_score(problem, ref_phase)
        if ref_score < best_score:
            best_phase = ref_phase
            best_overlap = ref_overlap
            best_score = ref_score
            best_mix = 1.0
            best_source = 'reference'
        if 'neural' in sources and neural_init is not None:
            ref_blend = self.cfg.reference_phase_mix * ref_phase + (1.0 - self.cfg.reference_phase_mix) * neural_init
            candidate_phases['reference_blend'] = ref_blend
            ref_overlap = self._estimate_overlap(problem, ref_blend)
            ref_score = self._estimate_init_score(problem, ref_blend)
            if ref_score < best_score:
                best_phase = ref_blend
                best_overlap = ref_overlap
                best_score = ref_score
                best_mix = self.cfg.reference_phase_mix
                best_source = 'reference_blend'
    if 'lin2025' in sources:
        try:
            lin_phase = predict_hologram_phase(
                problem['target_amp'], problem['target_phase'], problem['weight'],
                self.cfg, device=device,
            )
            candidate_phases['lin2025'] = lin_phase
            lin_overlap = self._estimate_overlap(problem, lin_phase)
            lin_score = self._estimate_init_score(problem, lin_phase)
            if lin_score < best_score:
                best_phase = lin_phase
                best_overlap = lin_overlap
                best_score = lin_score
                best_mix = 1.0
                best_source = 'lin2025'
        except Exception as e:
            print(f'lin2025 init failed: {e}')
    if 'warm_start' in sources and self.previous_phase is not None:
        warm = self.previous_phase
        if warm.shape != (self.cfg.slm_size, self.cfg.slm_size):
            warm = F.interpolate(warm.unsqueeze(0).unsqueeze(0), size=(self.cfg.slm_size, self.cfg.slm_size), mode='bilinear').squeeze(0).squeeze(0)
        if warm.device != device:
            warm = warm.to(device)
        candidate_phases['warm_start'] = warm
        warm_overlap = self._estimate_overlap(problem, warm)
        warm_score = self._estimate_init_score(problem, warm)
        if warm_score < best_score:
            best_phase = warm
            best_overlap = warm_overlap
            best_score = warm_score
            best_mix = 1.0
            best_source = 'warm_start'
    if best_phase is None:
        best_phase = torch.zeros((self.cfg.slm_size, self.cfg.slm_size), device=device)
        best_overlap = 0.0
        best_score = float('inf')
    self.last_init_source = best_source
    return best_phase, best_overlap, best_score

HybridHolographyRunner._build_init_phase = _fixed_build_init_phase
print('Fixed _build_init_phase: pure reference phase used when neural not in sources.')


## 训练 `Lin2025HologramNet`
这是当前学习型初始化器的主训练步骤，使用更大的 Colab 参数。


In [ ]:
from pathlib import Path
import gc
import json

from lin2025_holography.config import Lin2025Config
from lin2025_holography.training import train_lin2025_model


def cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUALITY_CHECKPOINT_SUBDIR = "wider_net_v44_integrated"
lin_ckpt_dir = Path("lin2025_holography") / "checkpoints" / QUALITY_CHECKPOINT_SUBDIR
lin_ckpt_dir.mkdir(parents=True, exist_ok=True)

base_kwargs = dict(
    target_mode="flat_top",
    slm_size=256,
    device=DEVICE,
    checkpoint_dir=lin_ckpt_dir,
    train_samples=320,
    val_samples=40,
    epochs=12,
    batch_size=8,
    learning_rate=2e-4,
    input_channels=4,
    hidden_channels=64,
    num_residual_blocks=6,
    teacher_weight_795=0.3,
    teacher_weight_hybrid=0.7,
    hybrid_teacher_phase_gate=0.0018,
    flat_overlap_weight=0.1,
    flat_uniformity_weight=0.25,
    flat_core_uniformity_weight=1.10,
    flat_efficiency_weight=0.60,
    flat_core_phase_weight=2.0,
    flat_phase_weight=0.1,
    flat_core_threshold=0.78,
    teacher_sources=("795", "hybrid"),
    teacher_weight_795_round=1.0,
    teacher_weight_hybrid_round=0.0,
    teacher_weight_795_flat=0.15,
    teacher_weight_hybrid_flat=0.85,
    hologram_core_phase_imitation_weight=1.1,
    hologram_roi_phase_imitation_weight=0.35,
    efficiency_floor=0.30,
)

fallbacks = [
    ("default", {}),
    ("batch4_full", {"batch_size": 4}),
    ("batch2_full", {"batch_size": 2}),
    ("batch2_mid", {"batch_size": 2, "hidden_channels": 48, "num_residual_blocks": 5, "train_samples": 256, "val_samples": 32}),
    ("safe", {"batch_size": 2, "hidden_channels": 32, "num_residual_blocks": 4, "train_samples": 160, "val_samples": 20}),
]

best_lin = None
last_oom = None
selected_profile = None

for profile_name, overrides in fallbacks:
    cleanup_cuda()
    kwargs = dict(base_kwargs)
    kwargs.update(overrides)
    lin_cfg = Lin2025Config(**kwargs)
    print(f"trying profile={profile_name} | batch_size={lin_cfg.batch_size} | hidden={lin_cfg.hidden_channels} | blocks={lin_cfg.num_residual_blocks} | train_samples={lin_cfg.train_samples} | val_samples={lin_cfg.val_samples}")
    try:
        best_lin = train_lin2025_model(lin_cfg)
        selected_profile = profile_name
        break
    except RuntimeError as e:
        message = str(e).lower()
        if "out of memory" in message or "cuda error" in message:
            print(f"profile {profile_name} failed due to GPU memory/runtime issue; retrying with a smaller configuration.")
            cleanup_cuda()
            last_oom = e
            continue
        raise

if best_lin is None:
    raise last_oom if last_oom is not None else RuntimeError("Training did not start successfully.")

print("selected training profile:", selected_profile)
print("lin2025 best checkpoint:", best_lin)

history_path = lin_ckpt_dir / "lin2025_training_history.json"
history = json.loads(history_path.read_text(encoding="utf-8"))
print("best supervised:", history["best_supervised_checkpoint"])
print("best quality:", history["best_quality_checkpoint"])
print("best hybrid polish:", history["best_hybrid_polish_checkpoint"])


## AutoDL 运行说明
1. 从上到下依次运行单元格。
2. 第一格会自动补装缺失的基础库，但不会重复安装 PyTorch。请先在 AutoDL 后台预置 GPU 版 PyTorch，推荐 `torch 2.8.0`，并确保 CUDA 为 12.8。
3. notebook 会优先使用 `/root/autodl-tmp/flat_top_light_runtime_v58/`，如果不可写再退到 `/tmp/` 或当前目录。
4. `Lin2025HologramNet` 训练完成后会生成 `colab_export_bundle.zip`，可直接下载回本地。
5. 如需继续看 benchmark，请在导出之后再运行后面的分析和可视化单元。


## 导出训练结果
这一步会导出一个 `colab_export_bundle/` 文件夹，并自动打包成 zip。\n\n你下载这个 zip 后，回到本地：\n1. 解压到一个新的 checkpoint 目录，例如 `lin2025_holography/checkpoints/colab_exp_xxx/`；\n2. 运行本地分析脚本 `colab_local_split/analyze_lin2025_checkpoint_local.py`；\n3. 不建议只用 `.npy` 作为中继，`.pt` checkpoint 才能支持本地完整分析。


In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import torch

from ai_holography.config import HolographyConfig
from ai_holography.pipeline import AIHolographyPipeline
from ai_holography.references import (
    apply_795_reference_metadata,
    find_795_reference_files,
    parse_795_reference_name,
)
from lin2025_holography.config import Lin2025Config
from lin2025_holography.predictor import predict_hologram_phase


EXPORT_DIR = Path("colab_export_bundle")
SUMMARY_PATH = Path("ai_holography") / "runs" / "benchmark_lin2025_hybrid" / "summary.json"
QUALITY_CHECKPOINT_SUBDIR = globals().get("QUALITY_CHECKPOINT_SUBDIR", "wider_net_v44_integrated")
lin_ckpt_dir = Path("lin2025_holography") / "checkpoints" / QUALITY_CHECKPOINT_SUBDIR

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for name in [
    "lin2025_best_supervised.pt",
    "lin2025_best_quality.pt",
    "lin2025_best_hybrid_polish.pt",
    "lin2025_training_history.json",
]:
    src = lin_ckpt_dir / name
    if src.exists():
        shutil.copy2(src, EXPORT_DIR / name)

refs = find_795_reference_files(Path("ftl_gen"))
device = "cuda" if torch.cuda.is_available() else "cpu"
lin_cfg = Lin2025Config(
    target_mode="flat_top",
    slm_size=256,
    device=device,
    checkpoint_dir=lin_ckpt_dir,
)


def build_problem_from_ref(style: str):
    for ref in refs:
        info = parse_795_reference_name(ref)
        if info.get("style") == style:
            cfg = HolographyConfig(run_name=f"export_{style}")
            cfg.device = device
            cfg.reference_phase_path = ref
            cfg = apply_795_reference_metadata(cfg, info)
            cfg.phase_type = "phase_flat"
            pipe = AIHolographyPipeline(cfg)
            return ref, pipe.build_problem()
    raise FileNotFoundError(style)


manifest = {
    "notebook_version": "v58",
    "project_root": str(Path.cwd()),
    "checkpoint_dir": str(lin_ckpt_dir),
    "export_dir": str(EXPORT_DIR),
    "files": [],
    "placement_instructions": {
        "local_checkpoint_dir": "D:/Trae products/flat_top light/lin2025_holography/checkpoints/<your_autodl_run_name>/",
        "local_analysis_script": "D:/Trae products/flat_top light/colab_local_split/analyze_lin2025_checkpoint_local.py",
    },
}

prediction_targets = [
    ("round_top", "lin2025_best_supervised.pt"),
    ("flat_top", "lin2025_best_supervised.pt"),
]

for style, checkpoint_name in prediction_targets:
    ref, problem = build_problem_from_ref(style)
    checkpoint_path = lin_ckpt_dir / checkpoint_name
    if not checkpoint_path.exists():
        continue
    pred_phase, loaded = predict_hologram_phase(
        lin_cfg,
        target_amp=problem["target_amp"],
        target_phase=problem["target_phase"],
        checkpoint=checkpoint_path,
    )
    out_name = f"pred_{style}_phase.npy"
    np.save(EXPORT_DIR / out_name, pred_phase.detach().cpu().numpy().astype(np.float32))
    manifest["files"].append(
        {
            "name": out_name,
            "source_reference": str(ref),
            "checkpoint_used": str(loaded) if loaded is not None else str(checkpoint_path),
        }
    )

if SUMMARY_PATH.exists():
    shutil.copy2(SUMMARY_PATH, EXPORT_DIR / "benchmark_summary.json")
    manifest["files"].append({"name": "benchmark_summary.json", "source": str(SUMMARY_PATH)})

(EXPORT_DIR / "export_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
zip_path = shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=EXPORT_DIR.parent, base_dir=EXPORT_DIR.name)
print("exported bundle dir:", EXPORT_DIR)
print("exported zip:", zip_path)
print("bundle contents:")
for p in sorted(EXPORT_DIR.iterdir()):
    print("-", p.name)


## 可选：跑统一 benchmark
会比较 `795_only`、`795_plus_hybrid`、`lin2025_plus_hybrid_balanced`、`lin2025_plus_hybrid_quality`、`lin2025_plus_feedback_proxy`。


In [ ]:
import json
from pathlib import Path
import numpy as np
import torch
import ai_holography.camera_loop as _camera_loop_mod
import ai_holography.runner as _runner_mod

from ai_holography.config import HolographyConfig
from ai_holography.losses import composite_loss
from ai_holography.profiles import apply_profile
from ai_holography.references import apply_795_reference_metadata, find_795_reference_files, parse_795_reference_name
from ai_holography.runner import HybridHolographyRunner
from ai_holography.visualization import save_field_visualizations, save_linecuts


def _camera_feedback_weight_device_safe(measured_intensity, target_intensity):
    measured = measured_intensity.to(target_intensity.device)
    measured = measured / measured.mean().clamp_min(1e-9)
    target = target_intensity / target_intensity.mean().clamp_min(1e-9)
    return target / measured.clamp_min(1e-6)


_camera_loop_mod.camera_feedback_weight = _camera_feedback_weight_device_safe
_runner_mod.camera_feedback_weight = _camera_feedback_weight_device_safe


def _select_reference(refs, style):
    for ref in refs:
        info = parse_795_reference_name(ref)
        if style == "flat_top":
            if info.get("style") == style and info.get("sx") is not None and info.get("sy") is not None:
                return ref
        elif info.get("style") == style:
            return ref
    raise FileNotFoundError(style)


def _base_cfg(ref: Path, run_name: str) -> HolographyConfig:
    cfg = apply_profile(HolographyConfig(run_name=run_name), "experiment")
    info = parse_795_reference_name(ref)
    cfg.reference_phase_path = ref
    cfg = apply_795_reference_metadata(cfg, info)
    cfg.phase_type = "phase_flat"
    cfg.uniformity_weight = 0.2
    cfg.efficiency_weight = 0.1
    cfg.core_uniformity_weight = 0.45
    cfg.core_phase_weight = 0.2
    cfg.core_region_threshold = 0.78
    cfg.polish_uniformity_weight = 0.18
    cfg.polish_efficiency_weight = 0.1
    cfg.polish_phase_weight = 0.1
    cfg.target_overlap = 0.9990
    cfg.hybrid_cg_maxiter = 35
    cfg.warm_start_candidate_threshold = 0.0
    cfg.warm_start_similarity_threshold = 0.0
    cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
    return cfg


def _evaluate_direct(cfg: HolographyConfig, phase: torch.Tensor, label: str):
    pipeline = HybridHolographyRunner(cfg).pipeline
    problem = pipeline.build_problem()
    out_amp, out_phase = pipeline.propagator(problem["beam"], phase.to(pipeline.device))
    loss, aux = composite_loss(
        target_amp=problem["target_amp"],
        target_phase=problem["target_phase"],
        out_amp=out_amp,
        out_phase=out_phase,
        weight=problem["weight"],
        slm_phase=phase.to(pipeline.device),
        efficiency_floor=cfg.efficiency_floor,
        overlap_weight=cfg.polish_overlap_weight,
        intensity_weight=cfg.polish_intensity_weight,
        phase_weight=cfg.polish_phase_weight,
        uniformity_weight=cfg.polish_uniformity_weight,
        efficiency_weight=cfg.polish_efficiency_weight,
        core_uniformity_weight=cfg.core_uniformity_weight,
        core_phase_weight=cfg.core_phase_weight,
        core_region_threshold=cfg.core_region_threshold,
        smoothness_weight=cfg.polish_smoothness_weight,
    )
    output_dir = cfg.output_dir / label
    output_dir.mkdir(parents=True, exist_ok=True)
    np.save(output_dir / "slm_phase.npy", phase.detach().cpu().numpy())
    metrics = {"composite_loss": float(loss.detach().cpu()), **aux}
    (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    save_field_visualizations(output_dir, problem["target_amp"], problem["target_phase"], out_amp, out_phase, phase)
    save_linecuts(output_dir, problem["target_amp"], out_amp)
    return metrics


def _route_score(metrics):
    return (
        2.0 * metrics.get("core_phase_flatness", float("inf"))
        + 600.0 * metrics.get("core_uniformity_loss", float("inf"))
        + 5000.0 * metrics.get("overlap_loss", float("inf"))
    )


def _rank_key(metrics):
    core_phase = metrics.get("core_phase_flatness", float("inf"))
    core_uniformity = metrics.get("core_uniformity_loss", float("inf"))
    overlap = 1.0 - metrics.get("overlap", 0.0)
    return core_phase, core_uniformity, overlap

def _configure_lin_hybrid_route(cfg, mode):
    cfg.init_candidate_sources = ("lin2025", "reference")
    cfg.lin2025_primary = True
    if mode == "balanced":
        cfg.reference_phase_mix = 0.25
        cfg.lin2025_reference_weak_mix = 0.25
    elif mode == "quality":
        cfg.reference_phase_mix = 0.35
        cfg.lin2025_reference_weak_mix = 0.35
    return cfg


def _run_lin2025_hybrid_route(round_ref: Path, flat_ref: Path, mode, summary: dict):
    route_name = f"lin2025_plus_hybrid_{mode}"
    round_cfg = _base_cfg(round_ref, f"benchmark_lin2025_{mode}_round")
    round_cfg = _configure_lin_hybrid_route(round_cfg, mode)
    round_runner = HybridHolographyRunner(round_cfg)
    round_phase, _ = round_runner.run_once(use_warm_start=False, save_subdir=f"round_{route_name}")

    flat_cfg = _base_cfg(flat_ref, f"benchmark_lin2025_{mode}_flat")
    flat_cfg = _configure_lin_hybrid_route(flat_cfg, mode)
    flat_runner = HybridHolographyRunner(flat_cfg)
    flat_runner.previous_phase = round_phase
    flat_runner.previous_target_amp = round_runner.previous_target_amp
    flat_runner.previous_target_phase = round_runner.previous_target_phase
    _, metrics = flat_runner.run_once(use_warm_start=True, save_subdir=f"flat_{route_name}")
    metrics["direction"] = f"lin2025_hybrid_{mode}"
    summary["routes"][route_name] = metrics


refs = find_795_reference_files(Path("ftl_gen"))
flat_ref = _select_reference(refs, "flat_top")
round_ref = _select_reference(refs, "round_top")

summary = {"routes": {}, "notebook_version": "v58", "quality_checkpoint_subdir": QUALITY_CHECKPOINT_SUBDIR}



def _run_reference_hybrid_route(round_ref: Path, flat_ref: Path, summary: dict):
    route_name = "795_plus_hybrid"
    round_cfg = _base_cfg(round_ref, "benchmark_795_hybrid_round")
    round_cfg.init_candidate_sources = ("reference",)
    round_runner = HybridHolographyRunner(round_cfg)
    round_phase, _ = round_runner.run_once(use_warm_start=False, save_subdir=f"round_{route_name}")

    flat_cfg = _base_cfg(flat_ref, "benchmark_795_hybrid_flat")
    flat_cfg.init_candidate_sources = ("reference", "warm_start")
    flat_runner = HybridHolographyRunner(flat_cfg)
    flat_runner.previous_phase = round_phase
    flat_runner.previous_target_amp = round_runner.previous_target_amp
    flat_runner.previous_target_phase = round_runner.previous_target_phase
    _, metrics = flat_runner.run_once(use_warm_start=True, save_subdir=f"flat_{route_name}")
    metrics["direction"] = "physical_reference_hybrid"
    summary["routes"][route_name] = metrics


def _run_feedback_proxy_route(round_ref: Path, flat_ref: Path, summary: dict):
    route_name = "lin2025_plus_feedback_proxy"
    round_cfg = _base_cfg(round_ref, "benchmark_feedback_proxy_round")
    round_cfg.init_candidate_sources = ("lin2025", "reference")
    round_cfg.lin2025_primary = True
    round_cfg.reference_phase_mix = 0.35
    round_cfg.lin2025_reference_weak_mix = 0.35
    round_runner = HybridHolographyRunner(round_cfg)
    round_phase, _ = round_runner.run_once(use_warm_start=False, save_subdir=f"round_{route_name}")

    flat_cfg = _base_cfg(flat_ref, "benchmark_feedback_proxy_flat")
    flat_cfg = _configure_lin_hybrid_route(flat_cfg, mode="quality")
    flat_cfg.feedback_target_gain = 0.08
    flat_runner = HybridHolographyRunner(flat_cfg)
    flat_runner.previous_phase = round_phase
    flat_runner.previous_target_amp = round_runner.previous_target_amp
    flat_runner.previous_target_phase = round_runner.previous_target_phase

    phase0, metrics0 = flat_runner.run_once(use_warm_start=True, save_subdir=f"flat_{route_name}_round0")
    output_dir0 = flat_cfg.output_dir / f"flat_{route_name}_round0"
    problem = flat_runner.pipeline.build_problem()
    phase_tensor = torch.tensor(np.load(output_dir0 / "slm_phase.npy"), dtype=torch.float32, device=flat_runner.pipeline.device)
    out_amp, _ = flat_runner.pipeline.propagator(problem["beam"], phase_tensor)
    measurement_file = flat_cfg.output_dir / f"{route_name}_measured.npy"
    measurement_file.parent.mkdir(parents=True, exist_ok=True)
    measured_i = out_amp.square()
    target_i = problem["target_amp"].square().to(measured_i.device)
    proxy_i = 0.60 * measured_i + 0.40 * target_i
    np.save(measurement_file, proxy_i.detach().cpu().numpy())

    flat_runner.camera_loop.enabled = True
    flat_runner.camera_loop.measured_intensity_path = str(measurement_file)
    _, metrics1 = flat_runner.run_once(use_warm_start=True, save_subdir=f"flat_{route_name}_round1")
    flat_runner.camera_loop.enabled = False
    flat_runner.camera_loop.measured_intensity_path = None
    metrics1["feedback_proxy_used"] = True
    metrics1["feedback_proxy_measurement"] = str(measurement_file)
    metrics1["feedback_proxy_measurement_blend"] = {"measured": 0.60, "target": 0.40}
    metrics1["feedback_proxy_delta_overlap"] = float(metrics1.get("overlap", 0.0) - metrics0.get("overlap", 0.0))
    metrics1["feedback_proxy_delta_efficiency"] = float(metrics1.get("efficiency", 0.0) - metrics0.get("efficiency", 0.0))
    metrics1["feedback_proxy_delta_core_uniformity_loss"] = float(metrics1.get("core_uniformity_loss", 0.0) - metrics0.get("core_uniformity_loss", 0.0))
    metrics1["feedback_proxy_delta_core_phase_flatness"] = float(metrics1.get("core_phase_flatness", 0.0) - metrics0.get("core_phase_flatness", 0.0))
    summary["routes"][route_name] = metrics1


refs = find_795_reference_files(Path("ftl_gen"))
flat_ref = _select_reference(refs, "flat_top")
round_ref = _select_reference(refs, "round_top")

summary = {"routes": {}, "notebook_version": "v58", "quality_checkpoint_subdir": QUALITY_CHECKPOINT_SUBDIR}



In [ ]:
# 795 only (best direct initialization without hybrid polish)
from ai_holography.references import load_reference_phase

round_cfg = _base_cfg(round_ref, "benchmark_795_round")
round_runner = HybridHolographyRunner(round_cfg)
round_problem = round_runner.pipeline.build_problem()
# Directly load reference phase instead of using _build_init_phase
round_phase = load_reference_phase(
    round_ref,
    size=(round_cfg.slm_size, round_cfg.slm_size),
    device=round_cfg.device,
)
round_runner.previous_phase = round_phase
round_runner.previous_target_amp = round_problem["target_amp"]
round_runner.previous_target_phase = round_problem["target_phase"]
_ = _evaluate_direct(round_cfg, round_phase, "round_795_only")

flat_cfg = _base_cfg(flat_ref, "benchmark_795_flat")
flat_runner = HybridHolographyRunner(flat_cfg)
# Directly load reference phase instead of using _build_init_phase
flat_phase = load_reference_phase(
    flat_ref,
    size=(flat_cfg.slm_size, flat_cfg.slm_size),
    device=flat_cfg.device,
)
summary["routes"]["795_only"] = _evaluate_direct(flat_cfg, flat_phase, "flat_795_only")

_run_reference_hybrid_route(round_ref, flat_ref, summary=summary)
_run_lin2025_hybrid_route(round_ref, flat_ref, mode="balanced", summary=summary)
_run_lin2025_hybrid_route(round_ref, flat_ref, mode="quality", summary=summary)
_run_feedback_proxy_route(round_ref, flat_ref, summary=summary)

primary_routes = {name: metrics for name, metrics in summary["routes"].items() if not metrics.get("exploratory_route", False)}
ranked = sorted(primary_routes.items(), key=lambda item: _rank_key(item[1]))
ranked_all = sorted(summary["routes"].items(), key=lambda item: _rank_key(item[1]))
summary["exploratory_routes"] = [name for name, metrics in summary["routes"].items() if metrics.get("exploratory_route", False)]
summary["ranking"] = [
    {"route": name, "rank_key": [float(value) for value in _rank_key(metrics)], "score": _route_score(metrics), "metrics": metrics}
    for name, metrics in ranked
]
summary["best_route"] = ranked[0][0] if ranked else None
summary["best_route_reason"] = (
    f"rank(core_phase, core_uniformity, overlap_loss)=({_rank_key(ranked[0][1])[0]:.6f}, {_rank_key(ranked[0][1])[1]:.6f}, {_rank_key(ranked[0][1])[2]:.6f}), diagnostic_score={_route_score(ranked[0][1]):.6f}"
    if ranked else None
)
summary["ranking_policy"] = "core_phase_flatness -> core_uniformity_loss -> overlap_loss -> diagnostic_score"

print('=== Route Summary ===')
print('ranking policy: core_phase_flatness -> core_uniformity_loss -> overlap_loss -> diagnostic_score')
print()
for name, metrics in summary["routes"].items():
    if metrics.get("exploratory_route", False):
        continue
    print(f'[{name}]')
    print(f'  overlap: {metrics.get("overlap", 0.0)}')
    print(f'  core_uniformity_loss: {metrics.get("core_uniformity_loss", 0.0)}')
    print(f'  core_phase_flatness: {metrics.get("core_phase_flatness", 0.0)}')
    print(f'  efficiency: {metrics.get("efficiency", 0.0)}')
    if 'quality_variant' in metrics:
        print(f'  quality_variant: {metrics.get("quality_variant")}')
    if 'selected_checkpoint' in metrics:
        print(f'  selected_checkpoint: {metrics.get("selected_checkpoint")}')
    print()
print('=== Ranking ===')
for i, (name, metrics) in enumerate(ranked, 1):
    print(f'{i}. {name} | score={_route_score(metrics):.6f}')
print()
if summary.get("exploratory_routes"):
    print('Exploratory routes excluded from BEST ROUTE:', ', '.join(summary["exploratory_routes"]))
    print()
print('=== Best Route ===')
if summary.get("best_route"):
    best_metrics = summary["routes"][summary["best_route"]]
    print(f'BEST ROUTE = {summary["best_route"]}')
    print(f'BEST ROUTE REASON = rank(core_phase, core_uniformity, overlap_loss)=({_rank_key(best_metrics)[0]:.6f}, {_rank_key(best_metrics)[1]:.6f}, {_rank_key(best_metrics)[2]:.6f}), diagnostic_score={_route_score(best_metrics):.6f}')
    print(f'  overlap: {best_metrics.get("overlap", 0.0)}')
    print(f'  core_uniformity_loss: {best_metrics.get("core_uniformity_loss", 0.0)}')
    print(f'  core_phase_flatness: {best_metrics.get("core_phase_flatness", 0.0)}')
    print(f'  efficiency: {best_metrics.get("efficiency", 0.0)}')
    print(f'  efficiency_floor: {best_metrics.get("efficiency_floor")}')
    print(f'  below_efficiency_floor: {best_metrics.get("below_efficiency_floor")}')

benchmark_dir = Path("ai_holography") / "runs" / "benchmark_lin2025_hybrid"
benchmark_dir.mkdir(parents=True, exist_ok=True)
summary_path = benchmark_dir / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Summary saved to {summary_path}")


## 分析 quality 路由结果
基于 notebook 和 `summary.json` 分析结果
1. 按照 `core_phase_flatness -> core_uniformity_loss -> overlap_loss` 排序路由
2. quality 路由使用了哪个 checkpoint?
3. 为什么这个路由被选为 quality 路由?
4. efficiency 路由的表现如何?


In [ ]:
from pathlib import Path
import json


def route_score(metrics: dict) -> float:
    return (
        2.0 * float(metrics.get("core_phase_flatness", float("inf")))
        + 0.6 * float(metrics.get("core_uniformity_loss", float("inf")))
        + 0.1 * float(metrics.get("overlap_loss", float("inf")))
    )


def route_rank(metrics: dict) -> tuple[float, float, float, float]:
    return (
        float(metrics.get("core_phase_flatness", float("inf"))),
        float(metrics.get("core_uniformity_loss", float("inf"))),
        float(metrics.get("overlap_loss", float("inf"))),
        route_score(metrics),
    )


def _checkpoint_name(path_value: str | None) -> str:
    return Path(path_value).name if path_value else "default"


def tuning_recommendations(summary: dict) -> dict:
    routes = summary.get("routes", {})
    best_route = summary.get("best_route")
    best_metrics = routes.get(best_route, {})
    baseline_metrics = routes.get("795_only", {})

    observations = []
    suggestions = []

    if best_route:
        rank = route_rank(best_metrics)
        observations.append(
            f"best route is {best_route}: rank=({rank[0]:.6f}, {rank[1]:.6f}, {rank[2]:.6f}), "
            f"diagnostic_score={rank[3]:.6f}."
        )

    if best_route and best_route.endswith("_quality"):
        observations.append(
            f"quality mainline is active with variant={best_metrics.get('quality_variant', 'baseline')} "
            f"and checkpoint={_checkpoint_name(best_metrics.get('selected_checkpoint'))}."
        )

    if baseline_metrics:
        observations.append(
            "vs 795_only baseline: "
            f"delta core_phase_flatness={float(best_metrics.get('core_phase_flatness', float('inf'))) - float(baseline_metrics.get('core_phase_flatness', float('inf'))):+.6f}, "
            f"delta core_uniformity={float(best_metrics.get('core_uniformity_loss', float('inf'))) - float(baseline_metrics.get('core_uniformity_loss', float('inf'))):+.6f}, "
            f"delta overlap={float(best_metrics.get('overlap', 0.0)) - float(baseline_metrics.get('overlap', 0.0)):+.6f}."
        )

    observations.append(
        "efficiency remains a diagnostic metric only in this benchmark and should not override core phase or core uniformity decisions."
    )

    best_core_phase = float(best_metrics.get("core_phase_flatness", float("inf")))
    baseline_core_phase = float(baseline_metrics.get("core_phase_flatness", float("inf")))
    best_core_uniformity = float(best_metrics.get("core_uniformity_loss", float("inf")))
    baseline_core_uniformity = float(baseline_metrics.get("core_uniformity_loss", float("inf")))
    quality_variant = best_metrics.get("quality_variant", "baseline")

    if baseline_metrics and best_core_phase >= baseline_core_phase:
        suggestions.append(
            {
                "priority": 1,
                "action": "increase phase bias inside the quality family",
                "recommended_changes": {
                    "phase_priority_core_phase_weight": [1.9, 2.1, 2.3],
                    "stage3_maxiter": [12, 14, 16],
                    "init_score_core_phase_weight": [2.4, 2.8, 3.2],
                },
                "why": "the winning route no longer beats the 795 baseline on core_phase_flatness, so the next sweep should push phase harder before touching anything else.",
            }
        )

    if baseline_metrics and best_core_phase < baseline_core_phase and best_core_uniformity > baseline_core_uniformity + 0.0025:
        suggestions.append(
            {
                "priority": 2,
                "action": "recover core uniformity inside the current quality variant",
                "recommended_changes": {
                    "core_uniformity_weight": [0.30, 0.34, 0.38],
                    "phase_post_correction_core_uniformity_weight": [0.04, 0.06, 0.08],
                    "polish_uniformity_weight": [0.02, 0.03, 0.04],
                },
                "why": "core phase is already ahead, so the next clean gain should come from reclaiming core uniformity without giving phase back.",
            }
        )

    if best_route and best_route.endswith("_quality") and quality_variant == "baseline":
        suggestions.append(
            {
                "priority": 1,
                "action": "promote the internal quality sweep to the next comparison run",
                "recommended_values": ["baseline_tightphase", "phase_first"],
                "why": "the baseline quality preset is still winning, so the next step should stay close to the current v30 winner and only test the nearest stronger-phase challengers.",
            }
        )
    elif best_route and best_route.endswith("_quality"):
        suggestions.append(
            {
                "priority": 1,
                "action": "refine locally around the current winning quality variant",
                "recommended_changes": {
                    "baseline_tightphase": {
                        "core_phase_weight": [1.18, 1.22, 1.26],
                        "phase_post_correction_core_phase_weight": [1.00, 1.10, 1.20],
                    },
                    "phase_first": {
                        "stage3_maxiter": [10, 12, 14],
                        "polish_phase_priority_phase_weight": [0.28, 0.30, 0.32],
                    },
                },
                "why": "the winning route already sits inside the quality family, so the next sweep should stay local rather than switching to balanced or feedback_proxy.",
            }
        )

    if not suggestions:
        suggestions.append(
            {
                "priority": 3,
                "action": "continue quality-local sweeps",
                "why": "the current winner already matches the phase-first objective, and no stronger single-metric regression is visible right now.",
            }
        )

    suggestions.sort(key=lambda x: int(x.get("priority", 99)))
    return {"observations": observations, "suggestions": suggestions}


summary_path = Path("ai_holography") / "runs" / "benchmark_lin2025_hybrid" / "summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
analysis = tuning_recommendations(summary)
summary["autodl_post_analysis"] = analysis
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(analysis, ensure_ascii=False, indent=2))


## 效率上限探测
在当前硬件+几何配置下，只优化 efficiency + overlap（不管相位平坦度），看 efficiency 的物理天花板是多少。
运行后把 `efficiency_floor` 设成 `0.85 × 上限` 即可。


In [ ]:
import torch
import torch.nn.functional as F
from ai_holography.config import HolographyConfig
from ai_holography.propagation import FourierSLM, wrap_phase
from ai_holography.targets import build_target, build_weight, threshold_weight, gaussian_beam
from ai_holography.losses import efficiency_metric, normalized_overlap

cfg_eff = HolographyConfig()
cfg_eff.device = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(cfg_eff.device)

propagator = FourierSLM(cfg_eff.slm_size, cfg_eff.output_size).to(device)
beam = gaussian_beam(cfg_eff.slm_size, cfg_eff.beam_sigma_x, cfg_eff.beam_sigma_y, cfg_eff.device)
center = (cfg_eff.output_size / 2.0, cfg_eff.output_size / 2.0)
target_amp = build_target("target_flat_top", cfg_eff.output_size, center, cfg_eff.target_sigma, 1, cfg_eff.device)
target_phase = torch.zeros((cfg_eff.output_size, cfg_eff.output_size), device=device)
roi = build_weight("gaussian_top_round", cfg_eff.output_size, center, cfg_eff.roi_diameter, cfg_eff.roi_softness, cfg_eff.device)
weight = threshold_weight(roi, fraction=cfg_eff.weight_threshold)
target_amp = target_amp * weight
target_amp = target_amp * torch.sqrt(beam.square().sum() / target_amp.square().sum().clamp_min(1e-9))

phase = torch.nn.Parameter(2.0 * torch.pi * torch.rand((cfg_eff.slm_size, cfg_eff.slm_size), device=device))
optimizer = torch.optim.AdamW([phase], lr=0.03)

best_eff = 0.0
for step in range(300):
    optimizer.zero_grad()
    out_amp, out_phase = propagator(beam, phase)
    overlap = normalized_overlap(target_amp, target_phase, out_amp, out_phase, weight)
    eff = efficiency_metric(weight, out_amp)
    loss = (1.0 - overlap).square() + 0.5 * (1.0 - eff).square()
    loss.backward()
    optimizer.step()
    if eff.item() > best_eff:
        best_eff = eff.item()
    if step % 50 == 0:
        print(f"step {step:3d}: overlap={overlap.item():.6f}  efficiency={eff.item():.6f}  best_eff={best_eff:.6f}")

print(f"\n=== Efficiency Ceiling ===")
print(f"max efficiency reached: {best_eff:.4f}")
print(f"suggested efficiency_floor (0.85 * ceiling): {0.85 * best_eff:.4f}")


## 查看结果
如果你只想把训练结果带回本地分析，到上一步导出 zip 就可以停止。\n下面这一步只是给 Colab 内部直接查看 benchmark 用。


In [ ]:
from pathlib import Path
import json

summary_path = Path("ai_holography") / "runs" / "benchmark_lin2025_hybrid" / "summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
routes = summary.get("routes", {})

order = [
    "795_plus_hybrid",
    "lin2025_plus_hybrid_balanced",
    "lin2025_plus_hybrid_quality",
    "lin2025_plus_feedback_proxy",
    "795_only",
]


def _checkpoint_name(path_value: str | None) -> str:
    return Path(path_value).name if path_value else "default"


print("=== Route Summary ===")
print("ranking policy:", summary.get("ranking_policy", "core_phase_flatness -> core_uniformity_loss -> overlap_loss"))
for name in order:
    metrics = routes.get(name)
    if not metrics:
        continue
    print()
    print(f"[{name}]")
    if "quality_variant" in metrics:
        print(f"  quality_variant: {metrics['quality_variant']}")
        print(f"  selected_checkpoint: {_checkpoint_name(metrics.get('selected_checkpoint'))}")
    for key in [
        "overlap",
        "core_uniformity_loss",
        "core_phase_flatness",
        "efficiency",
    ]:
        if key in metrics:
            print(f"  {key}: {metrics[key]}")

if "ranking" in summary:
    print()
    print("=== Ranking ===")
    for i, item in enumerate(summary["ranking"], start=1):
        print(f"{i}. {item['route']} | score={item['score']:.6f}")
if summary.get("exploratory_routes"):
    print()
    print("Exploratory routes excluded from BEST ROUTE:", ", ".join(summary["exploratory_routes"]))

best_route = summary["best_route"]
best_metrics = routes.get(best_route, {})
print()
print("=== Best Route ===")
print("BEST ROUTE =", best_route)
print("BEST ROUTE REASON =", summary.get("best_route_reason"))
print(f"  overlap: {best_metrics.get('overlap')}")
print(f"  core_uniformity_loss: {best_metrics.get('core_uniformity_loss')}")
print(f"  core_phase_flatness: {best_metrics.get('core_phase_flatness')}")
print(f"  efficiency: {best_metrics.get('efficiency')}")
print(f"  efficiency_floor: {best_metrics.get('efficiency_floor')}")
print(f"  below_efficiency_floor: {best_metrics.get('below_efficiency_floor')}")


In [ ]:
from pathlib import Path
import json
import math
import matplotlib.pyplot as plt
from IPython.display import Image, display

summary_path = Path("ai_holography") / "runs" / "benchmark_lin2025_hybrid" / "summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
routes = summary.get("routes", {})
benchmark_dir = summary_path.parent
plots_dir = benchmark_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)


def route_order(routes: dict) -> list[str]:
    preferred = [
        "795_only",
        "795_plus_hybrid",
        "lin2025_plus_hybrid_balanced",
        "lin2025_plus_hybrid_quality",
        "lin2025_plus_feedback_proxy",
    ]
    return [name for name in preferred if name in routes]


def route_output_dir(name: str) -> Path | None:
    if name == "lin2025_plus_hybrid_quality":
        metrics = routes.get(name, {})
        selected = metrics.get("selected_checkpoint")
        ckpt = Path(selected).stem if selected else None
        variant = metrics.get("quality_variant")
        if ckpt and variant:
            return Path("ai_holography") / "runs" / f"benchmark_quality_flat_{ckpt}_{variant}" / "outputs" / f"flat_lin2025_plus_hybrid_quality_{ckpt}_{variant}"
    fallback = {
        "795_only": Path("ai_holography") / "runs" / "benchmark_795_flat" / "outputs" / "flat_795_only",
        "795_plus_hybrid": Path("ai_holography") / "runs" / "benchmark_795_hybrid_flat" / "outputs" / "flat_795_plus_hybrid",
        "lin2025_plus_hybrid_balanced": Path("ai_holography") / "runs" / "benchmark_balanced_flat" / "outputs" / "flat_lin2025_plus_hybrid_balanced",
        "lin2025_plus_feedback_proxy": Path("ai_holography") / "runs" / "benchmark_feedback_proxy_flat" / "outputs" / "flat_lin2025_plus_feedback_proxy_round1",
        "lin2025_plus_hybrid_quality": Path("ai_holography") / "runs" / "benchmark_quality_flat" / "outputs" / "flat_lin2025_plus_hybrid_quality",
    }
    return fallback.get(name)


order = route_order(routes)
labels = {
    "795_only": "795 only",
    "795_plus_hybrid": "795+hybrid",
    "lin2025_plus_hybrid_balanced": "balanced",
    "lin2025_plus_hybrid_quality": "quality",
    "lin2025_plus_feedback_proxy": "feedback proxy",
}
core_uniformity = [float(routes[name].get("core_uniformity_loss", float("nan"))) for name in order]
core_phase = [float(routes[name].get("core_phase_flatness", float("nan"))) for name in order]
efficiency = [float(routes[name].get("efficiency", float("nan"))) for name in order]
efficiency_floor = float(summary.get("routes", {}).get(summary.get("best_route", ""), {}).get("efficiency_floor", 0.05))

x = list(range(len(order)))
fig, axes = plt.subplots(2, 1, figsize=(12, 10), constrained_layout=True)
width = 0.35
axes[0].bar([i - width / 2 for i in x], core_uniformity, width=width, label="Core uniformity loss", color="#5B4DB2")
axes[0].bar([i + width / 2 for i in x], core_phase, width=width, label="Core phase flatness", color="#1F9D8A")
axes[0].set_xticks(x, [labels[n] for n in order])
axes[0].set_ylabel("Loss (lower=better)")
axes[0].set_title("Route Comparison: Core Metrics")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend()

axes[1].bar(x, efficiency, width=0.55, color="#3A86E9", label="Efficiency")
axes[1].axhline(efficiency_floor, color="#D84A4A", linestyle="--", linewidth=1.5, label=f"Floor = {efficiency_floor:.3f}")
axes[1].set_xticks(x, [labels[n] for n in order])
axes[1].set_ylabel("Efficiency (higher=better)")
axes[1].set_ylim(0.0, max(1.0, max(efficiency + [efficiency_floor]) * 1.15))
axes[1].set_title("Route Comparison: Efficiency (diagnostic only)")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend()

metrics_plot = plots_dir / "route_metrics_overview.png"
fig.savefig(metrics_plot, dpi=180, bbox_inches="tight")
plt.close(fig)
print("saved:", metrics_plot)
display(Image(filename=str(metrics_plot)))

# 显示最佳路由的图片
print()
print("=== Best Route Images ===")
best_route = summary["best_route"]
out_dir = route_output_dir(best_route)
if out_dir is not None:
    for name in ["summary.png", "linecuts.png"]:
        p = out_dir / name
        if p.exists():
            display(Image(filename=str(p)))

existing = []
for name in order:
    base = route_output_dir(name)
    if base is None:
        continue
    summary_img = base / "summary.png"
    linecuts_img = base / "linecuts.png"
    if summary_img.exists() or linecuts_img.exists():
        existing.append((name, summary_img if summary_img.exists() else None, linecuts_img if linecuts_img.exists() else None))

for name, summary_img, linecuts_img in existing:
    label = labels[name]
    variant = routes.get(name, {}).get("quality_variant")
    if variant:
        label = f"{label} ({variant})"
    print()
    print(f"== {label} ==")
    if summary_img is not None:
        display(Image(filename=str(summary_img)))
    if linecuts_img is not None:
        display(Image(filename=str(linecuts_img)))
